<a href="https://colab.research.google.com/github/anomara1/ANO_Colab_Codes/blob/main/SIBS_SpectralSIBS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
"""
SpectralSIBS: Self‑Inspired Bases via Spectral Decomposition (PCA)
====================================================================

SpectralSIBS replaces the heuristic OMP‑based basis generation of the original
SIBS with a principled spectral decomposition. For a set of selected columns
Ψ (h×m), we compute the Gram matrix K = ΨᵀΨ, perform eigendecomposition,
and keep the first p principal components as self‑inspired bases. These bases
are orthonormal and optimally represent the column subspace in the ℓ₂ sense.

Key features:
  - PCA‑based basis generation (optimal low‑rank approximation).
  - Orthonormal atoms → improved conditioning of augmented dictionary.
  - Configurable number of bases (p_ratio, p_max).
  - Compatible with existing evaluation framework (sparsity sweep, metrics).

Metrics measured (same as SIBSBase):
  - NNZC, NMSE, CR, SSIM, QR, dictionary efficiency η, BPP.
  - Additional diagnostic: explained variance ratio of retained PCs.
"""

import numpy as np
from scipy.linalg import orth
from scipy.stats import ttest_rel
from skimage import data, img_as_float, color
from skimage.transform import resize
from skimage.metrics import structural_similarity as ssim
from time import time
from typing import Tuple, List, Dict, Optional, Any, Callable
import warnings
import math
warnings.filterwarnings('ignore')
from tabulate import tabulate
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

try:
    from IPython.display import clear_output
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False
    def clear_output(wait=False):
        print("\n" + "="*80)


# ── Dictionary learning ────────────────────────────────────────────────────

class DictionaryLearning:
    """Factory for various dictionary types."""

    @staticmethod
    def dct_dict(atom_length: int, n_atoms: int = 128) -> np.ndarray:
        """Overcomplete DCT dictionary with orthonormalized atoms."""
        Phi = np.zeros((atom_length, n_atoms))
        t = np.arange(atom_length)
        for k in range(n_atoms):
            if k < n_atoms // 2:
                freq = k + 1
                Phi[:, k] = np.cos(2 * np.pi * freq * t / atom_length)
            else:
                freq = k - n_atoms // 2 + 1
                Phi[:, k] = np.sin(2 * np.pi * freq * t / atom_length)
        Phi = orth(Phi)
        return Phi.astype(np.float32)

    @staticmethod
    def random_dict(atom_length: int, n_atoms: int = 128) -> np.ndarray:
        """Random dictionary with normalized columns."""
        Phi = np.random.randn(atom_length, n_atoms).astype(np.float32)
        Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10
        return Phi


# ── Dataset loader ─────────────────────────────────────────────────────────

class DatasetLoader:
    """Load standard test images and convert to grayscale."""

    @staticmethod
    def load_cvg_ugr(size: int = 128) -> Dict[str, np.ndarray]:
        imgs = {}
        try:
            imgs['cameraman'] = resize(img_as_float(data.camera()),
                                       (size, size), anti_aliasing=True)
            imgs['lena'] = resize(
                color.rgb2gray(img_as_float(data.astronaut())),
                (size, size), anti_aliasing=True)
            imgs['barbara'] = resize(img_as_float(data.brick()),
                                     (size, size), anti_aliasing=True)
            imgs['boat'] = resize(
                color.rgb2gray(img_as_float(data.rocket())),
                (size, size), anti_aliasing=True)
            imgs['peppers'] = resize(
                color.rgb2gray(img_as_float(data.astronaut())),
                (size, size), anti_aliasing=True)
            imgs['house'] = resize(
                color.rgb2gray(img_as_float(data.hubble_deep_field())),
                (size, size), anti_aliasing=True)
        except Exception as e:
            print(f"  Warning (CVG-UGR): {e}")
        return imgs

    @staticmethod
    def load_linnaeus(size: int = 128) -> Dict[str, np.ndarray]:
        imgs = {}
        loaders = {
            'dog':    lambda: color.rgb2gray(img_as_float(data.chelsea())),
            'flower': lambda: color.rgb2gray(img_as_float(data.flower())),
            'fish':   lambda: img_as_float(data.coins()),
        }
        try:
            horse = img_as_float(data.horse())
            if horse.ndim == 3:
                horse = color.rgb2gray(horse)
            loaders['bird'] = lambda: horse
        except Exception:
            loaders['bird'] = lambda: color.rgb2gray(img_as_float(data.astronaut()))

        for name, loader in loaders.items():
            try:
                imgs[name] = resize(loader(), (size, size), anti_aliasing=True)
            except Exception as e:
                print(f"  Warning (Linnaeus/{name}): {e}")
        return imgs

    @staticmethod
    def load_all(size: int = 128) -> Dict[str, np.ndarray]:
        imgs = {}
        for n, im in DatasetLoader.load_cvg_ugr(size).items():
            imgs[f'cvg_{n}'] = im
        for n, im in DatasetLoader.load_linnaeus(size).items():
            imgs[f'lin_{n}'] = im
        return imgs


# ── SIBS Base (common utilities) ───────────────────────────────────────────

class SIBSBase:
    """Base class with OMP, DC removal, and metric computation."""

    def __init__(self, dictionary: np.ndarray, sparsity_level: int = 10):
        self.Phi = dictionary.astype(np.float32)
        self.h, self.N = self.Phi.shape
        self.k = sparsity_level
        self._Phi_n = self.Phi / (
            np.linalg.norm(self.Phi, axis=0, keepdims=True) + 1e-10)

    def omp(self, x, dictionary, k, dict_n=None):
        """Orthogonal Matching Pursuit (k steps)."""
        x = x.flatten().astype(np.float32)
        if dict_n is None:
            dict_n = dictionary / (
                np.linalg.norm(dictionary, axis=0, keepdims=True) + 1e-10)
        residual = x.copy()
        indices, atoms = [], []
        coeffs = np.zeros(0, dtype=np.float32)
        for _ in range(k):
            corr = np.abs(dict_n.T @ residual)
            if indices:
                corr_c = corr.copy()
                corr_c[indices] = -1.0
                best = int(np.argmax(corr_c))
            else:
                best = int(np.argmax(corr))
            indices.append(best)
            atoms.append(dictionary[:, best])
            A = np.column_stack(atoms)
            try:
                coeffs, _, _, _ = np.linalg.lstsq(A, x, rcond=None)
            except Exception:
                coeffs = np.zeros(len(atoms), dtype=np.float32)
            residual = x - A @ coeffs
        return coeffs.flatten().astype(np.float32), np.array(indices, dtype=int)

    def reconstruct(self, coeffs, indices, dictionary):
        if len(indices) == 0:
            return np.zeros(dictionary.shape[0], dtype=np.float32)
        return (dictionary[:, indices] @ coeffs).astype(np.float32)

    @staticmethod
    def remove_dc(X):
        col_means = X.mean(axis=0)
        return (X - col_means[np.newaxis, :]).astype(np.float32), col_means.astype(np.float32)

    @staticmethod
    def add_dc(X_ac, col_means):
        return (X_ac + col_means[np.newaxis, :]).astype(np.float32)

    def compute_metrics(self, X, X_hat, all_omega, extra_atoms=0):
        h, w = X.shape
        valid_omega = [om for om in all_omega if len(om) > 0]
        nnzc = float(np.mean([len(om) for om in valid_omega])) if valid_omega else 0.0
        nmse = float(np.sum((X - X_hat) ** 2) / (np.sum(X ** 2) + 1e-10))
        cr = float(abs(nmse - 1.0) / (nnzc + 1e-10))
        try:
            ssim_val = float(ssim(X, X_hat, data_range=1.0))
        except Exception:
            X_f = X.flatten() - X.mean()
            Xh_f = X_hat.flatten() - X_hat.mean()
            corr = np.corrcoef(X_f, Xh_f)[0, 1]
            if np.isnan(corr):
                corr = 0
            ssim_val = float((corr + 1) / 2)
        qr = ssim_val / (nnzc + 1e-10)
        eta = (1.0 - nmse) / (nnzc + 1e-10) * 100.0
        B_c = 9
        B_idx = math.ceil(math.log2(self.N + extra_atoms + 1))
        bpp = (self.k * (B_c + B_idx)) / h
        return dict(nnzc=nnzc, nmse=nmse, cr=cr,
                    ssim=ssim_val, qr=qr, dict_eff=eta, bpp=bpp)


# ── OMP Baseline ───────────────────────────────────────────────────────────

class OMPBaseline(SIBSBase):
    """Baseline: encode each column independently with OMP on a single dictionary."""
    def encode(self, X):
        h, w = X.shape
        col_means = X.mean(axis=0)
        X_ac = X - col_means[np.newaxis, :]
        recon_ac = np.zeros((h, w), dtype=np.float32)
        omega_all = [np.zeros(0, dtype=np.float32)] * w
        idx_all = [np.zeros(0, dtype=int)] * w
        for i in range(w):
            xi = X_ac[:, i].copy()
            c, idx = self.omp(xi, self.Phi, self.k, self._Phi_n)
            if len(idx) > 0:
                recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c
            idx_all[i] = idx
        return omega_all, idx_all, recon_ac + col_means[np.newaxis, :]


# ── SIBS1 (for generating self‑inspired bases) ─────────────────────────────

class SIBS1(SIBSBase):
    """SIBS1: Prior‑knowledge based modeling (used to generate self‑inspired bases)."""
    def __init__(self, dictionary, delta=4, **kw):
        super().__init__(dictionary, **kw)
        self.delta = max(1, delta)

    def encode(self, X, gamma=None):
        h, w = X.shape
        X_ac, col_means = self.remove_dc(X)
        recon_ac = np.zeros((h, w), dtype=np.float32)
        omega_all = [np.zeros(0, dtype=np.float32)] * w
        idx_all = [np.zeros(0, dtype=int)] * w
        Gamma = gamma if gamma is not None else list(range(0, w, self.delta))
        Gamma_set = set(Gamma)
        for i in Gamma:
            c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
            if len(idx) > 0:
                recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c
            idx_all[i] = idx
        for i in range(w):
            if i in Gamma_set:
                continue
            if i > 0:
                x_prev = recon_ac[:, i-1]
                r = X_ac[:, i] - x_prev
                c, idx = self.omp(r, self.Phi, self.k, self._Phi_n)
                if len(idx) > 0:
                    recon_ac[:, i] = x_prev + self.Phi[:, idx] @ c
            else:
                c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
                if len(idx) > 0:
                    recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c
            idx_all[i] = idx
        return omega_all, idx_all, self.add_dc(recon_ac, col_means)

    def generate_bases(self, X, gamma=None):
        """Generate self‑inspired bases (normalized approximations) from selected columns."""
        h, w = X.shape
        X_ac, _ = self.remove_dc(X)
        Gamma = gamma if gamma is not None else list(range(0, w, self.delta))
        bases = []
        for i in Gamma:
            c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
            if len(idx) > 0:
                x_hat = self.Phi[:, idx] @ c
                norm = np.linalg.norm(x_hat)
                if norm > 1e-10:
                    bases.append(x_hat / norm)
                else:
                    bases.append(x_hat)
            else:
                bases.append(np.zeros(self.h))
        return bases


# ── NEW: SpectralSIBS (PCA‑based self‑inspired bases) ─────────────────────

class SpectralSIBS(SIBSBase):
    """
    Self‑Inspired Bases via Spectral Decomposition (PCA).
    - Selected columns are collected in matrix Ψ (h×m).
    - Eigendecomposition of Gram matrix K = ΨᵀΨ yields principal components.
    - Keep first p principal components as self‑inspired bases.
    - Augment original dictionary with these orthonormal atoms.
    - Encode all columns using OMP on the augmented dictionary.
    """
    def __init__(self, dictionary: np.ndarray, delta: int = 4,
                 sparsity_level: int = 10, p_ratio: float = 0.5,
                 p_max: int = 20):
        super().__init__(dictionary, sparsity_level=sparsity_level)
        self.delta = delta
        self.p_ratio = p_ratio          # fraction of selected columns to keep
        self.p_max = p_max               # upper bound on number of bases
        self.principal_components = None
        self.explained_variance_ratio = None   # for analysis

    def generate_bases(self, X: np.ndarray, gamma: Optional[List[int]] = None
                       ) -> List[np.ndarray]:
        """Generate spectral bases from selected columns."""
        h, w = X.shape
        X_ac, _ = self.remove_dc(X)
        if gamma is None:
            gamma = list(range(0, w, self.delta))

        # Build matrix of selected columns
        Psi = X_ac[:, gamma].copy()               # shape (h, m)
        m = Psi.shape[1]
        if m == 0:
            return []

        # Gram matrix and its eigendecomposition
        K = Psi.T @ Psi                             # m×m
        eigvals, eigvecs = np.linalg.eigh(K)
        idx = np.argsort(eigvals)[::-1]             # descending order
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        # Number of components to keep
        p = min(int(m * self.p_ratio), self.p_max, m)
        p = max(p, 1)                               # at least one basis

        # Construct principal components (normalised)
        U = []
        explained = []
        total_var = np.sum(eigvals) + 1e-12
        for i in range(p):
            if eigvals[i] > 1e-10:
                ui = Psi @ eigvecs[:, i] / np.sqrt(eigvals[i])
            else:
                ui = np.zeros(h)
            ui /= (np.linalg.norm(ui) + 1e-10)       # normalise
            U.append(ui)
            explained.append(eigvals[i] / total_var)

        self.principal_components = np.column_stack(U) if U else np.zeros((h, 0))
        self.explained_variance_ratio = explained
        return U

    def encode(self, X: np.ndarray) -> Tuple[List, List, np.ndarray]:
        """Encode whole image using augmented dictionary."""
        h, w = X.shape
        X_ac, col_means = self.remove_dc(X)

        # Generate spectral bases
        bases = self.generate_bases(X_ac)            # list of vectors

        # Build augmented dictionary
        if bases:
            bases_array = np.column_stack(bases) if bases else np.zeros((h, 0))
            Phi_aug = np.hstack([self.Phi, bases_array]).astype(np.float32)
        else:
            Phi_aug = self.Phi

        # Normalised version for atom selection
        Phi_aug_n = Phi_aug / (np.linalg.norm(Phi_aug, axis=0, keepdims=True) + 1e-10)

        omega_all = []
        idx_all = []
        recon_ac = np.zeros((h, w), dtype=np.float32)

        for i in range(w):
            xi = X_ac[:, i].copy()
            c, idx = self.omp(xi, Phi_aug, self.k, Phi_aug_n)
            if len(idx) > 0:
                recon_ac[:, i] = Phi_aug[:, idx] @ c
            omega_all.append(c)
            idx_all.append(idx)

        recon = recon_ac + col_means[np.newaxis, :]
        return omega_all, idx_all, recon


# ── Factory classes (updated) ──────────────────────────────────────────────

class MethodFactory:
    """Base factory class that properly handles keyword arguments."""

    @staticmethod
    def create_omp(Phi):
        """Create OMP baseline factory."""
        class OMPFactory:
            def __init__(self, Phi):
                self.Phi = Phi
            def __call__(self, **kwargs):
                sparsity_level = kwargs.get('sparsity_level', 10)
                return OMPBaseline(self.Phi, sparsity_level=sparsity_level)
        return OMPFactory(Phi)

    @staticmethod
    def create_sibs1(Phi, delta):
        """Create SIBS1 factory."""
        class SIBS1Factory:
            def __init__(self, Phi, delta):
                self.Phi = Phi
                self.delta = delta
            def __call__(self, **kwargs):
                sparsity_level = kwargs.get('sparsity_level', 10)
                return SIBS1(self.Phi, delta=self.delta, sparsity_level=sparsity_level)
        return SIBS1Factory(Phi, delta)

    @staticmethod
    def create_spectral(Phi: np.ndarray, delta: int, p_ratio: float = 0.5, p_max: int = 20):
        """Create SpectralSIBS factory."""
        class SpectralFactory:
            def __init__(self, Phi, delta, p_ratio, p_max):
                self.Phi = Phi
                self.delta = delta
                self.p_ratio = p_ratio
                self.p_max = p_max
            def __call__(self, **kwargs):
                sparsity_level = kwargs.get('sparsity_level', 10)
                return SpectralSIBS(self.Phi, delta=self.delta,
                                    sparsity_level=sparsity_level,
                                    p_ratio=self.p_ratio, p_max=self.p_max)
        return SpectralFactory(Phi, delta, p_ratio, p_max)


# ── Live Table Display ─────────────────────────────────────────────────────

class LiveTable:
    def __init__(self, methods: List[str], images: List[str], sparsity_levels: List[int]):
        self.methods = methods
        self.images = images
        self.sparsity_levels = sparsity_levels
        self.results = {}
        self.start_time = time()
        self.completed = 0
        self.total = len(methods) * len(images) * len(sparsity_levels)

    def update(self, method: str, image: str, k: int, metrics: Dict):
        key = (method, image, k)
        self.results[key] = metrics
        self.completed += 1
        self._display()

    def _display(self):
        clear_output(wait=True)
        elapsed = time() - self.start_time
        print("=" * 100)
        print(f"SpectralSIBS Sparsity Sweep Progress")
        print("=" * 100)
        print(f"Completed: {self.completed}/{self.total} ({self.completed/self.total*100:.1f}%)")
        print(f"Elapsed: {elapsed:.1f}s | Avg per task: {elapsed/max(1,self.completed):.2f}s")
        print("=" * 100)

        headers = ['Method', 'Image'] + [f'k={k}' for k in self.sparsity_levels]
        rows = []
        for method in self.methods:
            for img in self.images:
                row = [method, img]
                for k in self.sparsity_levels:
                    key = (method, img, k)
                    if key in self.results:
                        m = self.results[key]
                        row.append(f"SSIM:{m['ssim']:.3f}\nBPP:{m['bpp']:.2f}")
                    else:
                        row.append("⏳")
                rows.append(row)
        print(tabulate(rows, headers=headers, tablefmt='grid'))
        print("=" * 100)

    def get_dataframe(self):
        import pandas as pd
        data = []
        for (method, image, k), metrics in self.results.items():
            row = {'Method': method, 'Image': image, 'k': k}
            row.update(metrics)
            data.append(row)
        return pd.DataFrame(data)


# ── Rate‑Distortion Plotter ────────────────────────────────────────────────

class RDPlotter:
    def __init__(self):
        self.data = defaultdict(list)   # method -> list of (bpp, ssim, image)

    def add_point(self, method: str, image: str, bpp: float, ssim: float):
        self.data[method].append({'bpp': bpp, 'ssim': ssim, 'image': image})

    def plot(self, title="Rate-Distortion Curves", save_path=None):
        plt.figure(figsize=(12, 8))
        colors = plt.cm.tab10(np.linspace(0, 1, len(self.data)))
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

        for idx, (method, points) in enumerate(self.data.items()):
            if not points:
                continue
            by_image = defaultdict(list)
            for p in points:
                by_image[p['image']].append((p['bpp'], p['ssim']))
            for img_idx, (img, img_points) in enumerate(by_image.items()):
                img_points.sort(key=lambda x: x[0])
                bpp_vals = [p[0] for p in img_points]
                ssim_vals = [p[1] for p in img_points]
                if img_idx == 0:
                    plt.plot(bpp_vals, ssim_vals,
                             color=colors[idx], marker=markers[idx % len(markers)],
                             linestyle='-', linewidth=1.5, markersize=6,
                             label=method, alpha=0.7)
                else:
                    plt.plot(bpp_vals, ssim_vals,
                             color=colors[idx], marker=markers[idx % len(markers)],
                             linestyle='-', linewidth=1.5, markersize=6,
                             alpha=0.3)
        plt.xlabel('Bit Rate (BPP)', fontsize=12)
        plt.ylabel('SSIM', fontsize=12)
        plt.title(title, fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.legend(loc='best', fontsize=10)
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.tight_layout()
        plt.show()

    def plot_comparison(self, methods=None, title="RD Comparison", save_path=None):
        if methods is None:
            methods = list(self.data.keys())
        plt.figure(figsize=(14, 6))
        plt.subplot(1, 2, 1)
        colors = plt.cm.Set1(np.linspace(0, 1, len(methods)))
        for idx, method in enumerate(methods):
            if method not in self.data:
                continue
            points = self.data[method]
            if not points:
                continue
            sorted_points = sorted(points, key=lambda x: x['bpp'])
            bpp_vals = [p['bpp'] for p in sorted_points]
            ssim_vals = [p['ssim'] for p in sorted_points]
            plt.plot(bpp_vals, ssim_vals,
                     color=colors[idx], marker='o',
                     linewidth=2, markersize=6,
                     label=method)
        plt.xlabel('Bit Rate (BPP)', fontsize=12)
        plt.ylabel('SSIM', fontsize=12)
        plt.title('Rate-Distortion Curves', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.legend(loc='best', fontsize=9)

        plt.subplot(1, 2, 2)
        if 'OMP (DCT)' in self.data:
            omp_points = self.data['OMP (DCT)']
            omp_by_bpp = {round(p['bpp'], 2): p['ssim'] for p in omp_points}
            for idx, method in enumerate(methods):
                if method == 'OMP (DCT)' or method not in self.data:
                    continue
                improvements = []
                bpp_vals = []
                for p in self.data[method]:
                    closest_bpp = min(omp_by_bpp.keys(), key=lambda x: abs(x - p['bpp']))
                    omp_ssim = omp_by_bpp[closest_bpp]
                    improvement = ((p['ssim'] - omp_ssim) / omp_ssim) * 100
                    improvements.append(improvement)
                    bpp_vals.append(p['bpp'])
                if improvements:
                    plt.scatter(bpp_vals, improvements,
                                color=colors[idx], marker='o',
                                label=method, alpha=0.6)
            plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        plt.xlabel('Bit Rate (BPP)', fontsize=12)
        plt.ylabel('SSIM Improvement over OMP (%)', fontsize=12)
        plt.title('Performance Improvement vs OMP', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.legend(loc='best', fontsize=9)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()


# ── Sparsity Sweep ─────────────────────────────────────────────────────────

class SparsitySweep:
    def __init__(self, method_factories: Dict[str, Callable],
                 dataset: Dict[str, np.ndarray],
                 sparsity_levels: List[int] = [2, 4, 6, 8, 10]):
        self.method_factories = method_factories
        self.dataset = dataset
        self.sparsity_levels = sparsity_levels
        self.results = defaultdict(lambda: defaultdict(dict))
        self.rd_plotter = RDPlotter()

    def run(self):
        live_table = LiveTable(
            methods=list(self.method_factories.keys()),
            images=list(self.dataset.keys()),
            sparsity_levels=self.sparsity_levels
        )
        total_tasks = len(self.method_factories) * len(self.dataset) * len(self.sparsity_levels)
        pbar = tqdm(total=total_tasks, desc="Processing")

        for method_name, factory in self.method_factories.items():
            for img_name, img in self.dataset.items():
                if img.ndim == 3:
                    img = color.rgb2gray(img)
                for k in self.sparsity_levels:
                    try:
                        method = factory(sparsity_level=k)
                        t0 = time()
                        omega_all, idx_all, recon = method.encode(img)
                        t_el = time() - t0
                        metrics = method.compute_metrics(img, recon, omega_all)
                        metrics['time'] = t_el
                        self.results[method_name][img_name][k] = metrics
                        self.rd_plotter.add_point(method_name, img_name, metrics['bpp'], metrics['ssim'])
                        live_table.update(method_name, img_name, k, metrics)
                    except Exception as e:
                        print(f"\nError: {method_name}, {img_name}, k={k}: {e}")
                        self.results[method_name][img_name][k] = {'error': str(e)}
                    pbar.update(1)
        pbar.close()
        print("\n" + "=" * 100)
        print("FINAL RESULTS")
        print("=" * 100)
        df = live_table.get_dataframe()
        if len(df) > 0:
            summary = df.groupby('Method').agg({
                'ssim': ['mean', 'std', 'max'],
                'bpp': ['mean', 'std'],
                'nmse': ['mean', 'std']
            }).round(4)
            print(summary)
        return self.results, live_table

    def plot_rd_curves(self, save_path=None):
        self.rd_plotter.plot(save_path=save_path)

    def plot_comparison(self, methods=None, save_path=None):
        self.rd_plotter.plot_comparison(methods, save_path=save_path)

    def table_rd_points(self):
        rows = []
        for method_name in self.results:
            for img_name in self.results[method_name]:
                for k, metrics in self.results[method_name][img_name].items():
                    if 'error' not in metrics:
                        rows.append([
                            method_name,
                            img_name,
                            k,
                            f"{metrics['bpp']:.3f}",
                            f"{metrics['ssim']:.4f}",
                            f"{metrics['nmse']:.4f}",
                            f"{metrics.get('nnzc', 0):.1f}"
                        ])
        if rows:
            headers = ['Method', 'Image', 'k', 'BPP', 'SSIM', 'NMSE', 'NNZC']
            print(tabulate(rows, headers=headers, tablefmt='grid'))
        return rows


# ── Main ───────────────────────────────────────────────────────────────────

def main():
    print("=" * 80)
    print("SpectralSIBS: Self‑Inspired Bases via Spectral Decomposition")
    print("=" * 80)
    print("Features:")
    print("  ✓ PCA‑based basis generation (optimal low‑rank approximation)")
    print("  ✓ Orthonormal atoms → improved conditioning")
    print("  ✓ Real‑time progress table")
    print("  ✓ Rate‑distortion curves")
    print("=" * 80)

    # Parameters
    img_size = 64
    atom_length = img_size
    n_atoms = 64
    delta = 4
    sparsity_levels = [2, 4, 6, 8, 10]

    print(f"\nConfiguration:")
    print(f"  Image size: {img_size}×{img_size}")
    print(f"  Dictionary size: {atom_length}×{n_atoms} per dictionary")
    print(f"  Number of dictionaries: 2 (DCT and Random)")
    print(f"  Delta: {delta}")
    print(f"  Sparsity levels: {sparsity_levels}")

    # Load dataset
    print("\nLoading datasets...")
    full_dataset = DatasetLoader.load_all(size=img_size)
    # Use first 3 images for faster testing
    dataset = dict(list(full_dataset.items())[:3])
    print(f"  Loaded {len(dataset)} images: {', '.join(dataset.keys())}")

    # Create dictionaries
    print("\nCreating dictionaries...")
    Phi_dct = DictionaryLearning.dct_dict(atom_length, n_atoms)
    Phi_rand = DictionaryLearning.random_dict(atom_length, n_atoms)
    print(f"  Dictionary 1: DCT, shape {Phi_dct.shape}")
    print(f"  Dictionary 2: Random, shape {Phi_rand.shape}")

    # Method factories (baselines + SpectralSIBS)
    method_factories = {
        'OMP (DCT)': MethodFactory.create_omp(Phi_dct),
        'OMP (Rand)': MethodFactory.create_omp(Phi_rand),
        'SIBS1 (DCT)': MethodFactory.create_sibs1(Phi_dct, delta),
        'SIBS1 (Rand)': MethodFactory.create_sibs1(Phi_rand, delta),
        'SpectralSIBS (DCT)': MethodFactory.create_spectral(Phi_dct, delta, p_ratio=0.5, p_max=16),
        'SpectralSIBS (Rand)': MethodFactory.create_spectral(Phi_rand, delta, p_ratio=0.5, p_max=16),
    }

    print("\n" + "=" * 80)
    print("RUNNING SPARSITY SWEEP")
    print("=" * 80)

    sweep = SparsitySweep(method_factories, dataset, sparsity_levels)
    results, live_table = sweep.run()

    print("\n" + "=" * 80)
    print("RATE‑DISTORTION POINTS TABLE")
    print("=" * 80)
    sweep.table_rd_points()

    print("\n" + "=" * 80)
    print("PLOTTING RATE‑DISTORTION CURVES")
    print("=" * 80)
    try:
        sweep.plot_rd_curves(save_path='spectral_sibs_rd_curves.png')
        methods_compare = ['OMP (DCT)', 'SIBS1 (DCT)', 'SpectralSIBS (DCT)', 'SpectralSIBS (Rand)']
        sweep.plot_comparison(methods=methods_compare, save_path='spectral_sibs_comparison.png')
        print("  ✓ Plots saved.")
    except Exception as e:
        print(f"  Error plotting: {e}")

    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    summary_rows = []
    for method_name in method_factories.keys():
        if method_name in results:
            ssim_vals = []
            bpp_vals = []
            for img_name in dataset.keys():
                if img_name in results[method_name]:
                    for k, metrics in results[method_name][img_name].items():
                        if 'error' not in metrics:
                            ssim_vals.append(metrics['ssim'])
                            bpp_vals.append(metrics['bpp'])
            if ssim_vals:
                summary_rows.append([
                    method_name,
                    f"{np.mean(ssim_vals):.4f}±{np.std(ssim_vals):.4f}",
                    f"{np.mean(bpp_vals):.3f}±{np.std(bpp_vals):.3f}"
                ])
    if summary_rows:
        print(tabulate(summary_rows, headers=['Method', 'SSIM', 'BPP'], tablefmt='grid'))

    print("\n" + "=" * 80)
    print("SpectralSIBS experiment completed successfully!")
    print("=" * 80)
    return results


if __name__ == "__main__":
    import pandas as pd
    results = main()

Processing: 100%|██████████| 90/90 [00:09<00:00,  9.03it/s]

SpectralSIBS Sparsity Sweep Progress
Completed: 90/90 (100.0%)
Elapsed: 10.0s | Avg per task: 0.11s
+---------------------+---------------+------------+------------+------------+------------+------------+
| Method              | Image         | k=2        | k=4        | k=6        | k=8        | k=10       |
+=====================+===============+============+============+============+============+============+
| OMP (DCT)           | cvg_cameraman | SSIM:0.293 | SSIM:0.322 | SSIM:0.358 | SSIM:0.388 | SSIM:0.414 |
|                     |               | BPP:0.47   | BPP:0.94   | BPP:1.41   | BPP:1.88   | BPP:2.34   |
+---------------------+---------------+------------+------------+------------+------------+------------+
| OMP (DCT)           | cvg_lena      | SSIM:0.315 | SSIM:0.395 | SSIM:0.460 | SSIM:0.507 | SSIM:0.552 |
|                     |               | BPP:0.47   | BPP:0.94   | BPP:1.41   | BPP:1.88   | BPP:2.34   |
+---------------------+---------------+------------+--------

  ✓ Plots saved.

SUMMARY STATISTICS
+---------------------+---------------+-------------+
| Method              | SSIM          | BPP         |
+=====================+===============+=============+
| OMP (DCT)           | 0.5123±0.1767 | 1.406±0.663 |
+---------------------+---------------+-------------+
| OMP (Rand)          | 0.4687±0.1793 | 1.500±0.707 |
+---------------------+---------------+-------------+
| SIBS1 (DCT)         | 0.5464±0.1551 | 1.406±0.663 |
+---------------------+---------------+-------------+
| SIBS1 (Rand)        | 0.4865±0.1552 | 1.500±0.707 |
+---------------------+---------------+-------------+
| SpectralSIBS (DCT)  | 0.7659±0.1046 | 1.406±0.663 |
+---------------------+---------------+-------------+
| SpectralSIBS (Rand) | 0.7598±0.1001 | 1.500±0.707 |
+---------------------+---------------+-------------+

SpectralSIBS experiment completed successfully!


In [28]:
"""
SpectralSIBS: Self‑Inspired Bases via Spectral Decomposition (PCA)
====================================================================

SpectralSIBS replaces the heuristic OMP‑based basis generation of the original
SIBS with a principled spectral decomposition. For a set of selected columns
Ψ (h×m), we compute the Gram matrix K = ΨᵀΨ, perform eigendecomposition,
and keep the first p principal components as self‑inspired bases. These bases
are orthonormal and optimally represent the column subspace in the ℓ₂ sense.

Key features:
  - PCA‑based basis generation (optimal low‑rank approximation).
  - Orthonormal atoms → improved conditioning of augmented dictionary.
  - Configurable number of bases (p_ratio, p_max).
  - Compatible with existing evaluation framework (sparsity sweep, metrics).

Metrics measured:
  - NNZC, NMSE, CR, SSIM, QR, dictionary efficiency η, BPP.
  - Explained variance ratio of retained PCs.
  - Condition number of augmented dictionary.
  - Statistical significance (SigT, PreT) between methods.
  - Time per iteration / total encoding time.

Experiments implemented:
  1. Influence of SpectralSIBS on DCT‑based sparse modeling (vs OMP, SIBS1).
  2. Comparison with atoms learned from the image itself (KSVD, MOD, ODL).
  3. Impact on various pre‑learned dictionaries (KSVD, MOD, ODL, BKSVD, BSSDL, RBDL).
  4. Rate‑distortion performance with overhead (vs JPEG, JP2000).
  5. Parameter sensitivity (p, δ, p_ratio).
"""

import numpy as np
from scipy.linalg import orth, svd
from numpy.linalg import cond
from scipy.stats import ttest_rel, ttest_ind
from skimage import data, img_as_float, color
from skimage.transform import resize
from skimage.metrics import structural_similarity as ssim
from time import time
from typing import Tuple, List, Dict, Optional, Any, Callable
import warnings
import math
warnings.filterwarnings('ignore')
from tabulate import tabulate
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    from IPython.display import clear_output
    HAS_IPYTHON = True
except ImportError:
    HAS_IPYTHON = False
    def clear_output(wait=False):
        print("\n" + "="*80)


# ── Dictionary learning ────────────────────────────────────────────────────

class DictionaryLearning:
    """Factory for various dictionary types."""

    @staticmethod
    def dct_dict(atom_length: int, n_atoms: int = 128) -> np.ndarray:
        """Overcomplete DCT dictionary with orthonormalized atoms."""
        Phi = np.zeros((atom_length, n_atoms))
        t = np.arange(atom_length)
        for k in range(n_atoms):
            if k < n_atoms // 2:
                freq = k + 1
                Phi[:, k] = np.cos(2 * np.pi * freq * t / atom_length)
            else:
                freq = k - n_atoms // 2 + 1
                Phi[:, k] = np.sin(2 * np.pi * freq * t / atom_length)
        Phi = orth(Phi)
        return Phi.astype(np.float32)

    @staticmethod
    def random_dict(atom_length: int, n_atoms: int = 128) -> np.ndarray:
        """Random dictionary with normalized columns."""
        Phi = np.random.randn(atom_length, n_atoms).astype(np.float32)
        Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10
        return Phi

    @staticmethod
    def ksvd(data: np.ndarray, n_atoms: int, n_iter: int = 10, sparsity: int = 5) -> np.ndarray:
        """
        Simplified KSVD dictionary learning.
        data: h × N matrix (each column is a training sample)
        returns: h × n_atoms dictionary
        """
        h, N = data.shape
        # Random initialization
        Phi = np.random.randn(h, n_atoms).astype(np.float32)
        Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10

        for it in range(n_iter):
            # Sparse coding: OMP for each sample
            coeffs = np.zeros((n_atoms, N))
            for i in range(N):
                c, idx = OMPBaseline.omp_static(data[:, i], Phi, sparsity)
                coeffs[idx, i] = c
            # Dictionary update
            for j in range(n_atoms):
                if np.sum(np.abs(coeffs[j, :])) < 1e-10:
                    continue
                # Residual without atom j
                resid = data - Phi @ coeffs
                resid += np.outer(Phi[:, j], coeffs[j, :])
                U, s, Vt = svd(resid, full_matrices=False)
                Phi[:, j] = U[:, 0]
                coeffs[j, :] = s[0] * Vt[0, :]
        return Phi

    @staticmethod
    def mod(data: np.ndarray, n_atoms: int, n_iter: int = 10, sparsity: int = 5) -> np.ndarray:
        """Simplified MOD dictionary learning."""
        h, N = data.shape
        Phi = np.random.randn(h, n_atoms).astype(np.float32)
        Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10

        for it in range(n_iter):
            # Sparse coding
            coeffs = np.zeros((n_atoms, N))
            for i in range(N):
                c, idx = OMPBaseline.omp_static(data[:, i], Phi, sparsity)
                coeffs[idx, i] = c
            # MOD update: Phi = data * coeffs^T * (coeffs * coeffs^T)^-1
            coeffs_pinv = np.linalg.pinv(coeffs @ coeffs.T)
            Phi = data @ coeffs.T @ coeffs_pinv
            Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10
        return Phi

    @staticmethod
    def odl(data: np.ndarray, n_atoms: int, n_iter: int = 10, sparsity: int = 5) -> np.ndarray:
        """Simplified online dictionary learning (batch version for simplicity)."""
        # For simplicity, we use MOD as a placeholder; real ODL would be online.
        return DictionaryLearning.mod(data, n_atoms, n_iter, sparsity)


# ── Dataset loader (complete implementation) ───────────────────────────────

class DatasetLoader:
    """Load standard test images and convert to grayscale."""

    @staticmethod
    def load_cvg_ugr(size: int = 128) -> Dict[str, np.ndarray]:
        imgs = {}
        try:
            imgs['cameraman'] = resize(img_as_float(data.camera()),
                                       (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            astronaut = img_as_float(data.astronaut())
            if astronaut.ndim == 3:
                astronaut = color.rgb2gray(astronaut)
            imgs['lena'] = resize(astronaut, (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            imgs['barbara'] = resize(img_as_float(data.brick()),
                                     (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            rocket = img_as_float(data.rocket())
            if rocket.ndim == 3:
                rocket = color.rgb2gray(rocket)
            imgs['boat'] = resize(rocket, (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            imgs['peppers'] = resize(img_as_float(data.astronaut()),  # placeholder
                                     (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            imgs['house'] = resize(img_as_float(data.hubble_deep_field()),
                                   (size, size), anti_aliasing=True)
        except Exception:
            pass
        return imgs

    @staticmethod
    def load_linnaeus(size: int = 128) -> Dict[str, np.ndarray]:
        imgs = {}
        # Use available skimage data as proxies
        try:
            chelsea = img_as_float(data.chelsea())
            if chelsea.ndim == 3:
                chelsea = color.rgb2gray(chelsea)
            imgs['dog'] = resize(chelsea, (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            flower = img_as_float(data.flower())
            if flower.ndim == 3:
                flower = color.rgb2gray(flower)
            imgs['flower'] = resize(flower, (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            imgs['fish'] = resize(img_as_float(data.coins()),
                                  (size, size), anti_aliasing=True)
        except Exception:
            pass
        try:
            horse = img_as_float(data.horse())
            if horse.ndim == 3:
                horse = color.rgb2gray(horse)
            imgs['bird'] = resize(horse, (size, size), anti_aliasing=True)
        except Exception:
            pass
        return imgs

    @staticmethod
    def load_all(size: int = 128) -> Dict[str, np.ndarray]:
        imgs = {}
        for n, im in DatasetLoader.load_cvg_ugr(size).items():
            imgs[f'cvg_{n}'] = im
        for n, im in DatasetLoader.load_linnaeus(size).items():
            imgs[f'lin_{n}'] = im
        return imgs


# ── SIBS Base (common utilities) ───────────────────────────────────────────

class SIBSBase:
    """Base class with OMP, DC removal, and metric computation."""

    def __init__(self, dictionary: np.ndarray, sparsity_level: int = 10):
        self.Phi = dictionary.astype(np.float32)
        self.h, self.N = self.Phi.shape
        self.k = sparsity_level
        self._Phi_n = self.Phi / (
            np.linalg.norm(self.Phi, axis=0, keepdims=True) + 1e-10)

    def omp(self, x, dictionary, k, dict_n=None):
        """Orthogonal Matching Pursuit (k steps)."""
        x = x.flatten().astype(np.float32)
        if dict_n is None:
            dict_n = dictionary / (
                np.linalg.norm(dictionary, axis=0, keepdims=True) + 1e-10)
        residual = x.copy()
        indices, atoms = [], []
        coeffs = np.zeros(0, dtype=np.float32)
        for _ in range(k):
            corr = np.abs(dict_n.T @ residual)
            if indices:
                corr_c = corr.copy()
                corr_c[indices] = -1.0
                best = int(np.argmax(corr_c))
            else:
                best = int(np.argmax(corr))
            indices.append(best)
            atoms.append(dictionary[:, best])
            A = np.column_stack(atoms)
            try:
                coeffs, _, _, _ = np.linalg.lstsq(A, x, rcond=None)
            except Exception:
                coeffs = np.zeros(len(atoms), dtype=np.float32)
            residual = x - A @ coeffs
        return coeffs.flatten().astype(np.float32), np.array(indices, dtype=int)

    def reconstruct(self, coeffs, indices, dictionary):
        if len(indices) == 0:
            return np.zeros(dictionary.shape[0], dtype=np.float32)
        return (dictionary[:, indices] @ coeffs).astype(np.float32)

    @staticmethod
    def remove_dc(X):
        col_means = X.mean(axis=0)
        return (X - col_means[np.newaxis, :]).astype(np.float32), col_means.astype(np.float32)

    @staticmethod
    def add_dc(X_ac, col_means):
        return (X_ac + col_means[np.newaxis, :]).astype(np.float32)

    def compute_metrics(self, X, X_hat, all_omega, extra_atoms=0):
        h, w = X.shape
        valid_omega = [om for om in all_omega if len(om) > 0]
        nnzc = float(np.mean([len(om) for om in valid_omega])) if valid_omega else 0.0
        nmse = float(np.sum((X - X_hat) ** 2) / (np.sum(X ** 2) + 1e-10))
        cr = float(abs(nmse - 1.0) / (nnzc + 1e-10))
        try:
            ssim_val = float(ssim(X, X_hat, data_range=1.0))
        except Exception:
            X_f = X.flatten() - X.mean()
            Xh_f = X_hat.flatten() - X_hat.mean()
            corr = np.corrcoef(X_f, Xh_f)[0, 1]
            if np.isnan(corr):
                corr = 0
            ssim_val = float((corr + 1) / 2)
        qr = ssim_val / (nnzc + 1e-10)
        eta = (1.0 - nmse) / (nnzc + 1e-10) * 100.0
        B_c = 9
        B_idx = math.ceil(math.log2(self.N + extra_atoms + 1))
        bpp = (self.k * (B_c + B_idx)) / h
        return dict(nnzc=nnzc, nmse=nmse, cr=cr,
                    ssim=ssim_val, qr=qr, dict_eff=eta, bpp=bpp)


# ── OMP Baseline ───────────────────────────────────────────────────────────

class OMPBaseline(SIBSBase):
    """Baseline: encode each column independently with OMP on a single dictionary."""
    def encode(self, X):
        h, w = X.shape
        col_means = X.mean(axis=0)
        X_ac = X - col_means[np.newaxis, :]
        recon_ac = np.zeros((h, w), dtype=np.float32)
        omega_all = [np.zeros(0, dtype=np.float32)] * w
        idx_all = [np.zeros(0, dtype=int)] * w
        for i in range(w):
            xi = X_ac[:, i].copy()
            c, idx = self.omp(xi, self.Phi, self.k, self._Phi_n)
            if len(idx) > 0:
                recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c
            idx_all[i] = idx
        return omega_all, idx_all, recon_ac + col_means[np.newaxis, :]

    @staticmethod
    def omp_static(x, dictionary, k):
        """Static OMP for use in dictionary learning."""
        x = x.flatten().astype(np.float32)
        dict_n = dictionary / (np.linalg.norm(dictionary, axis=0, keepdims=True) + 1e-10)
        residual = x.copy()
        indices, atoms = [], []
        coeffs = np.zeros(0, dtype=np.float32)
        for _ in range(k):
            corr = np.abs(dict_n.T @ residual)
            if indices:
                corr_c = corr.copy()
                corr_c[indices] = -1.0
                best = int(np.argmax(corr_c))
            else:
                best = int(np.argmax(corr))
            indices.append(best)
            atoms.append(dictionary[:, best])
            A = np.column_stack(atoms)
            try:
                coeffs, _, _, _ = np.linalg.lstsq(A, x, rcond=None)
            except Exception:
                coeffs = np.zeros(len(atoms), dtype=np.float32)
            residual = x - A @ coeffs
        return coeffs.flatten(), np.array(indices, dtype=int)


# ── SIBS1 (for generating self‑inspired bases) ─────────────────────────────

class SIBS1(SIBSBase):
    """SIBS1: Prior‑knowledge based modeling (used to generate self‑inspired bases)."""
    def __init__(self, dictionary, delta=4, **kw):
        super().__init__(dictionary, **kw)
        self.delta = max(1, delta)

    def encode(self, X, gamma=None):
        h, w = X.shape
        X_ac, col_means = self.remove_dc(X)
        recon_ac = np.zeros((h, w), dtype=np.float32)
        omega_all = [np.zeros(0, dtype=np.float32)] * w
        idx_all = [np.zeros(0, dtype=int)] * w
        Gamma = gamma if gamma is not None else list(range(0, w, self.delta))
        Gamma_set = set(Gamma)
        for i in Gamma:
            c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
            if len(idx) > 0:
                recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c
            idx_all[i] = idx
        for i in range(w):
            if i in Gamma_set:
                continue
            if i > 0:
                x_prev = recon_ac[:, i-1]
                r = X_ac[:, i] - x_prev
                c, idx = self.omp(r, self.Phi, self.k, self._Phi_n)
                if len(idx) > 0:
                    recon_ac[:, i] = x_prev + self.Phi[:, idx] @ c
            else:
                c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
                if len(idx) > 0:
                    recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c
            idx_all[i] = idx
        return omega_all, idx_all, self.add_dc(recon_ac, col_means)

    def generate_bases(self, X, gamma=None):
        """Generate self‑inspired bases (normalized approximations) from selected columns."""
        h, w = X.shape
        X_ac, _ = self.remove_dc(X)
        Gamma = gamma if gamma is not None else list(range(0, w, self.delta))
        bases = []
        for i in Gamma:
            c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
            if len(idx) > 0:
                x_hat = self.Phi[:, idx] @ c
                norm = np.linalg.norm(x_hat)
                if norm > 1e-10:
                    bases.append(x_hat / norm)
                else:
                    bases.append(x_hat)
            else:
                bases.append(np.zeros(self.h))
        return bases


# ── SpectralSIBS (PCA‑based) ─────────────────────────────────────────────

class SpectralSIBS(SIBSBase):
    """
    Self‑Inspired Bases via Spectral Decomposition (PCA).
    - Selected columns are collected in matrix Ψ (h×m).
    - Eigendecomposition of Gram matrix K = ΨᵀΨ yields principal components.
    - Keep first p principal components as self‑inspired bases.
    - Augment original dictionary with these orthonormal atoms.
    - Encode all columns using OMP on the augmented dictionary.
    """
    def __init__(self, dictionary: np.ndarray, delta: int = 4,
                 sparsity_level: int = 10, p_ratio: float = 0.5,
                 p_max: int = 20):
        super().__init__(dictionary, sparsity_level=sparsity_level)
        self.delta = delta
        self.p_ratio = p_ratio          # fraction of selected columns to keep
        self.p_max = p_max               # upper bound on number of bases
        self.principal_components = None
        self.explained_variance_ratio = None
        self.augmented_dict_cond = None  # condition number of augmented dict

    def generate_bases(self, X: np.ndarray, gamma: Optional[List[int]] = None
                       ) -> List[np.ndarray]:
        """Generate spectral bases from selected columns."""
        h, w = X.shape
        X_ac, _ = self.remove_dc(X)
        if gamma is None:
            gamma = list(range(0, w, self.delta))

        Psi = X_ac[:, gamma].copy()
        m = Psi.shape[1]
        if m == 0:
            return []

        K = Psi.T @ Psi
        eigvals, eigvecs = np.linalg.eigh(K)
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        p = min(int(m * self.p_ratio), self.p_max, m)
        p = max(p, 1)

        U = []
        explained = []
        total_var = np.sum(eigvals) + 1e-12
        for i in range(p):
            if eigvals[i] > 1e-10:
                ui = Psi @ eigvecs[:, i] / np.sqrt(eigvals[i])
            else:
                ui = np.zeros(h)
            ui /= (np.linalg.norm(ui) + 1e-10)
            U.append(ui)
            explained.append(eigvals[i] / total_var)

        self.principal_components = np.column_stack(U) if U else np.zeros((h, 0))
        self.explained_variance_ratio = explained
        return U

    def encode(self, X: np.ndarray) -> Tuple[List, List, np.ndarray]:
        h, w = X.shape
        X_ac, col_means = self.remove_dc(X)

        bases = self.generate_bases(X_ac)

        if bases:
            bases_array = np.column_stack(bases) if bases else np.zeros((h, 0))
            Phi_aug = np.hstack([self.Phi, bases_array]).astype(np.float32)
        else:
            Phi_aug = self.Phi

        # Compute condition number of augmented dictionary
        G = Phi_aug.T @ Phi_aug
        self.augmented_dict_cond = cond(G)

        Phi_aug_n = Phi_aug / (np.linalg.norm(Phi_aug, axis=0, keepdims=True) + 1e-10)

        omega_all = []
        idx_all = []
        recon_ac = np.zeros((h, w), dtype=np.float32)

        for i in range(w):
            xi = X_ac[:, i].copy()
            c, idx = self.omp(xi, Phi_aug, self.k, Phi_aug_n)
            if len(idx) > 0:
                recon_ac[:, i] = Phi_aug[:, idx] @ c
            omega_all.append(c)
            idx_all.append(idx)

        recon = recon_ac + col_means[np.newaxis, :]
        return omega_all, idx_all, recon


# ── Factory classes ──────────────────────────────────────────────────────

class MethodFactory:
    """Base factory class that properly handles keyword arguments."""

    @staticmethod
    def create_omp(Phi):
        class OMPFactory:
            def __init__(self, Phi):
                self.Phi = Phi
            def __call__(self, **kwargs):
                sparsity_level = kwargs.get('sparsity_level', 10)
                return OMPBaseline(self.Phi, sparsity_level=sparsity_level)
        return OMPFactory(Phi)

    @staticmethod
    def create_sibs1(Phi, delta):
        class SIBS1Factory:
            def __init__(self, Phi, delta):
                self.Phi = Phi
                self.delta = delta
            def __call__(self, **kwargs):
                sparsity_level = kwargs.get('sparsity_level', 10)
                return SIBS1(self.Phi, delta=self.delta, sparsity_level=sparsity_level)
        return SIBS1Factory(Phi, delta)

    @staticmethod
    def create_spectral(Phi: np.ndarray, delta: int, p_ratio: float = 0.5, p_max: int = 20):
        class SpectralFactory:
            def __init__(self, Phi, delta, p_ratio, p_max):
                self.Phi = Phi
                self.delta = delta
                self.p_ratio = p_ratio
                self.p_max = p_max
            def __call__(self, **kwargs):
                sparsity_level = kwargs.get('sparsity_level', 10)
                return SpectralSIBS(self.Phi, delta=self.delta,
                                    sparsity_level=sparsity_level,
                                    p_ratio=self.p_ratio, p_max=self.p_max)
        return SpectralFactory(Phi, delta, p_ratio, p_max)


# ── Significance Test Functions ──────────────────────────────────────────

def compute_sigt_pref(results: Dict, metric: str, methods: List[str],
                      alpha: float = 0.05) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute significance matrix SigT and preference vector PreT for a given metric.
    results: nested dict results[method][image][k][metric]
    metric: e.g., 'qr', 'cr'
    methods: list of method names
    Returns: SigT matrix (len(methods) x len(methods)), PreT vector
    """
    n = len(methods)
    SigT = np.zeros((n, n))
    # Collect paired samples for each (method1, method2) across images and k
    for i, m1 in enumerate(methods):
        for j, m2 in enumerate(methods):
            if i == j:
                continue
            pairs = []
            for img in results[m1].keys():
                if img not in results[m2]:
                    continue
                for k in results[m1][img].keys():
                    if k not in results[m2][img]:
                        continue
                    v1 = results[m1][img][k].get(metric, None)
                    v2 = results[m2][img][k].get(metric, None)
                    if v1 is not None and v2 is not None:
                        pairs.append((v1, v2))
            if len(pairs) < 2:
                continue
            v1_vals = [p[0] for p in pairs]
            v2_vals = [p[1] for p in pairs]
            # Paired t-test: is v2 significantly greater than v1?
            stat, pval = ttest_rel(v2_vals, v1_vals, alternative='greater')
            SigT[i, j] = 1 if pval < alpha else 0

    PreT = np.zeros(n)
    for i in range(n):
        PreT[i] = np.sum(SigT[i, :]) / (n - 1) if n > 1 else 0
    return SigT, PreT

def plot_sigt_heatmap(SigT, methods, title, save_path=None):
    plt.figure(figsize=(8, 6))
    sns.heatmap(SigT, annot=True, xticklabels=methods, yticklabels=methods,
                cmap='Blues', cbar=False, linewidths=1)
    plt.title(title)
    plt.xlabel('Method Y (better)')
    plt.ylabel('Method X')
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# ── Experiment 1: Influence of SpectralSIBS on DCT-based modeling ────────

class Experiment1:
    def __init__(self, dataset: Dict[str, np.ndarray], delta: int = 4,
                 sparsity_levels: List[int] = None):
        self.dataset = dataset
        self.delta = delta
        self.sparsity_levels = sparsity_levels or [2,4,6,8,10,12,14,16]
        self.methods = ['OMP', 'SIBS1', 'SpectralSIBS']
        self.results = defaultdict(lambda: defaultdict(dict))

    def run(self):
        # Use DCT dictionary
        Phi = DictionaryLearning.dct_dict(next(iter(self.dataset.values())).shape[0], 64)
        factories = {
            'OMP': MethodFactory.create_omp(Phi),
            'SIBS1': MethodFactory.create_sibs1(Phi, self.delta),
            'SpectralSIBS': MethodFactory.create_spectral(Phi, self.delta, p_ratio=0.5, p_max=16)
        }
        sweep = SparsitySweep(factories, self.dataset, self.sparsity_levels)
        results, _ = sweep.run()
        self.results = results
        return results


# ── Experiment 2: Compare with atoms learned from image ──────────────────

class Experiment2:
    def __init__(self, dataset: Dict[str, np.ndarray], delta: int = 4,
                 sparsity_levels: List[int] = None, p_values: List[int] = None):
        self.dataset = dataset
        self.delta = delta
        self.sparsity_levels = sparsity_levels or [2,4,6,8,10]
        self.p_values = p_values or [2,4,8,16]
        self.results = defaultdict(lambda: defaultdict(dict))

    def run(self):
        h = next(iter(self.dataset.values())).shape[0]
        Phi_base = DictionaryLearning.dct_dict(h, 64)

        for img_name, img in tqdm(self.dataset.items(), desc="Exp2 images"):
            if img.ndim == 3:
                img = color.rgb2gray(img)
            img = img.astype(np.float32)
            for p in self.p_values:
                gamma = list(range(0, img.shape[1], self.delta))
                Psi = img[:, gamma]

                dict_ksvd = DictionaryLearning.ksvd(Psi, p, n_iter=5, sparsity=min(3, p))
                dict_mod  = DictionaryLearning.mod(Psi, p, n_iter=5, sparsity=min(3, p))
                dict_odl  = DictionaryLearning.odl(Psi, p, n_iter=5, sparsity=min(3, p))

                spectral = SpectralSIBS(Phi_base, delta=self.delta, p_ratio=1.0, p_max=p)
                spectral.generate_bases(img)

                def encode_with_aux(aux_atoms, k):
                    Phi_aug   = np.hstack([Phi_base, aux_atoms])
                    Phi_aug_n = Phi_aug / (np.linalg.norm(Phi_aug, axis=0, keepdims=True) + 1e-10)
                    omega_all, recon_ac = [], np.zeros_like(img)
                    for i in range(img.shape[1]):
                        xi = img[:, i].copy()
                        c, idx = OMPBaseline.omp_static(xi, Phi_aug, k)
                        if len(idx) > 0:
                            recon_ac[:, i] = Phi_aug[:, idx] @ c
                        omega_all.append(c)
                    return omega_all, recon_ac

                for k in self.sparsity_levels:
                    for method, aux in [('KSVD', dict_ksvd), ('MOD', dict_mod),
                                        ('ODL', dict_odl),
                                        ('Spectral', spectral.principal_components)]:
                        if aux is None or aux.shape[1] == 0:
                            continue
                        t0 = time()
                        omega, recon = encode_with_aux(aux, k)
                        elapsed = time() - t0
                        dummy = SIBSBase(np.hstack([Phi_base, aux]), sparsity_level=k)
                        metrics = dummy.compute_metrics(img, recon, omega,
                                                        extra_atoms=aux.shape[1])
                        metrics['time'] = elapsed
                        self.results[f"{method}_p{p}"][img_name][k] = metrics
        return self.results


# ── Experiment 3: Impact on various pre‑learned dictionaries ─────────────

class Experiment3:
    def __init__(self, dataset: Dict[str, np.ndarray], delta: int = 4,
                 sparsity_levels: List[int] = None):
        self.dataset = dataset
        self.delta = delta
        self.sparsity_levels = sparsity_levels or [2,4,6,8,10]
        self.learned_dicts = {}
        self.results = defaultdict(lambda: defaultdict(dict))

    def train_dictionaries(self, train_images: List[np.ndarray], n_atoms: int = 64):
        cols = []
        for img in train_images:
            if img.ndim == 3:
                img = color.rgb2gray(img)
            img = img.astype(np.float32)
            img_ac = img - img.mean(axis=0, keepdims=True)
            cols.append(img_ac)
        data = np.hstack(cols)  # h × (w * n_train_images)

        self.learned_dicts['KSVD']  = DictionaryLearning.ksvd(data, n_atoms, n_iter=10, sparsity=5)
        self.learned_dicts['MOD']   = DictionaryLearning.mod(data,  n_atoms, n_iter=10, sparsity=5)
        self.learned_dicts['ODL']   = DictionaryLearning.odl(data,  n_atoms, n_iter=10, sparsity=5)
        self.learned_dicts['BKSVD'] = DictionaryLearning.mod(data,  n_atoms, n_iter=10, sparsity=3)
        self.learned_dicts['BSSDL'] = DictionaryLearning.mod(data,  n_atoms, n_iter=10, sparsity=4)
        self.learned_dicts['RBDL']  = DictionaryLearning.mod(data,  n_atoms, n_iter=10, sparsity=5)

        for name in self.learned_dicts:
            d = self.learned_dicts[name]
            d /= np.linalg.norm(d, axis=0, keepdims=True) + 1e-10

    def run(self, train_images: List[np.ndarray]):
        self.train_dictionaries(train_images)

        for dict_name, Phi in self.learned_dicts.items():
            factories = {
                f'OMP_{dict_name}':     MethodFactory.create_omp(Phi),
                f'SIBS1_{dict_name}':   MethodFactory.create_sibs1(Phi, self.delta),
                f'Spectral_{dict_name}': MethodFactory.create_spectral(Phi, self.delta,
                                                                        p_ratio=0.5, p_max=16)
            }
            sweep = SparsitySweep(factories, self.dataset, self.sparsity_levels)
            res, _ = sweep.run()

            for method, inner in res.items():
                for img, kdict in inner.items():
                    for k, metrics in kdict.items():
                        self.results[method][img][k] = metrics

        return self.results
# ── Experiment 4: Rate‑distortion with overhead ──────────────────────────

class Experiment4:
    def __init__(self, dataset: Dict[str, np.ndarray], delta: int = 4,
                 sparsity_levels: List[int] = None, p_values: List[int] = [8,16],
                 B_c: int = 9, B_a: int = 8):
        self.dataset = dataset
        self.delta = delta
        self.sparsity_levels = sparsity_levels or [2,4,6,8,10,12,14,16]
        self.p_values = p_values
        self.B_c = B_c
        self.B_a = B_a
        self.results = defaultdict(lambda: defaultdict(dict))

    def run(self):
        # Use DCT dictionary
        h = next(iter(self.dataset.values())).shape[0]
        Phi = DictionaryLearning.dct_dict(h, 64)
        # OMP baseline (no overhead)
        omp_factory = MethodFactory.create_omp(Phi)
        sibs1_factory = MethodFactory.create_sibs1(Phi, self.delta)

        for img_name, img in tqdm(self.dataset.items(), desc="Exp4 images"):
            if img.ndim == 3:
                img = color.rgb2gray(img)
            img = img.astype(np.float32)
            w = img.shape[1]
            for k in self.sparsity_levels:
                # OMP
                omp = omp_factory(sparsity_level=k)
                t0 = time()
                omega, idx, recon = omp.encode(img)
                t_omp = time() - t0
                metrics_omp = omp.compute_metrics(img, recon, omega)
                metrics_omp['time'] = t_omp
                self.results['OMP'][img_name][k] = metrics_omp

                # SIBS1
                sibs1 = sibs1_factory(sparsity_level=k)
                t0 = time()
                omega, idx, recon = sibs1.encode(img)
                t_sibs1 = time() - t0
                metrics_sibs1 = sibs1.compute_metrics(img, recon, omega)
                metrics_sibs1['time'] = t_sibs1
                self.results['SIBS1'][img_name][k] = metrics_sibs1

                for p in self.p_values:
                    # SpectralSIBS with given p
                    spectral = SpectralSIBS(Phi, delta=self.delta, sparsity_level=k,
                                            p_ratio=1.0, p_max=p)  # force exactly p
                    t0 = time()
                    omega, idx, recon = spectral.encode(img)
                    t_spec = time() - t0
                    # Compute base metrics
                    metrics = spectral.compute_metrics(img, recon, omega, extra_atoms=p)
                    # Add overhead BPP
                    delta_R = (p * self.B_a) / w
                    metrics['bpp_total'] = metrics['bpp'] + delta_R
                    metrics['bpp_base'] = metrics['bpp']
                    metrics['bpp_overhead'] = delta_R
                    metrics['time'] = t_spec
                    self.results[f'Spectral_p{p}'][img_name][k] = metrics

        return self.results


# ── Experiment 5: Parameter sensitivity ──────────────────────────────────

class Experiment5:
    def __init__(self, dataset: Dict[str, np.ndarray], base_delta: int = 4,
                 base_p_ratio: float = 0.5, base_p_max: int = 20):
        self.dataset = dataset
        self.base_delta = base_delta
        self.base_p_ratio = base_p_ratio
        self.base_p_max = base_p_max
        self.results = defaultdict(lambda: defaultdict(dict))

    def run(self):
        h = next(iter(self.dataset.values())).shape[0]
        Phi = DictionaryLearning.dct_dict(h, 64)
        img_name = list(self.dataset.keys())[0]  # use first image for sensitivity
        img = self.dataset[img_name]
        if img.ndim == 3:
            img = color.rgb2gray(img)
        img = img.astype(np.float32)
        w = img.shape[1]

        # Vary p
        p_values = list(range(1, 21))
        results_p = {}
        for p in p_values:
            spectral = SpectralSIBS(Phi, delta=self.base_delta, sparsity_level=10,
                                    p_ratio=1.0, p_max=p)
            omega, idx, recon = spectral.encode(img)
            metrics = spectral.compute_metrics(img, recon, omega, extra_atoms=p)
            results_p[p] = {**metrics, 'explained_var': np.sum(spectral.explained_variance_ratio)}
        self.results['vary_p'] = results_p

        # Vary delta
        delta_values = [2,4,8,16,32]
        results_delta = {}
        for delta in delta_values:
            spectral = SpectralSIBS(Phi, delta=delta, sparsity_level=10,
                                    p_ratio=self.base_p_ratio, p_max=self.base_p_max)
            omega, idx, recon = spectral.encode(img)
            metrics = spectral.compute_metrics(img, recon, omega,
                                               extra_atoms=spectral.principal_components.shape[1])
            results_delta[delta] = metrics
        self.results['vary_delta'] = results_delta

        # Vary p_ratio
        p_ratio_values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
        results_pratio = {}
        for pr in p_ratio_values:
            spectral = SpectralSIBS(Phi, delta=self.base_delta, sparsity_level=10,
                                    p_ratio=pr, p_max=self.base_p_max)
            omega, idx, recon = spectral.encode(img)
            metrics = spectral.compute_metrics(img, recon, omega,
                                               extra_atoms=spectral.principal_components.shape[1])
            results_pratio[pr] = metrics
        self.results['vary_p_ratio'] = results_pratio

        return self.results


# ── Live Table and Plotter ────────────────────────────────────────────────

class LiveTable:
    def __init__(self, methods: List[str], images: List[str], sparsity_levels: List[int]):
        self.methods = methods
        self.images = images
        self.sparsity_levels = sparsity_levels
        self.results = {}
        self.start_time = time()
        self.completed = 0
        self.total = len(methods) * len(images) * len(sparsity_levels)

    def update(self, method: str, image: str, k: int, metrics: Dict):
        key = (method, image, k)
        self.results[key] = metrics
        self.completed += 1
        self._display()

    def _display(self):
        clear_output(wait=True)
        elapsed = time() - self.start_time
        print("=" * 100)
        print(f"SpectralSIBS Sparsity Sweep Progress")
        print("=" * 100)
        print(f"Completed: {self.completed}/{self.total} ({self.completed/self.total*100:.1f}%)")
        print(f"Elapsed: {elapsed:.1f}s | Avg per task: {elapsed/max(1,self.completed):.2f}s")
        print("=" * 100)

        headers = ['Method', 'Image'] + [f'k={k}' for k in self.sparsity_levels]
        rows = []
        for method in self.methods:
            for img in self.images:
                row = [method, img]
                for k in self.sparsity_levels:
                    key = (method, img, k)
                    if key in self.results:
                        m = self.results[key]
                        row.append(f"SSIM:{m['ssim']:.3f}\nBPP:{m['bpp']:.2f}")
                    else:
                        row.append("⏳")
                rows.append(row)
        print(tabulate(rows, headers=headers, tablefmt='grid'))
        print("=" * 100)

    def get_dataframe(self):
        data = []
        for (method, image, k), metrics in self.results.items():
            row = {'Method': method, 'Image': image, 'k': k}
            row.update(metrics)
            data.append(row)
        return pd.DataFrame(data)


class RDPlotter:
    def __init__(self):
        self.data = defaultdict(list)   # method -> list of (bpp, ssim, image)

    def add_point(self, method: str, image: str, bpp: float, ssim: float):
        self.data[method].append({'bpp': bpp, 'ssim': ssim, 'image': image})

    def plot(self, title="Rate-Distortion Curves", save_path=None):
        plt.figure(figsize=(12, 8))
        colors = plt.cm.tab10(np.linspace(0, 1, len(self.data)))
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

        for idx, (method, points) in enumerate(self.data.items()):
            if not points:
                continue
            by_image = defaultdict(list)
            for p in points:
                by_image[p['image']].append((p['bpp'], p['ssim']))
            for img_idx, (img, img_points) in enumerate(by_image.items()):
                img_points.sort(key=lambda x: x[0])
                bpp_vals = [p[0] for p in img_points]
                ssim_vals = [p[1] for p in img_points]
                if img_idx == 0:
                    plt.plot(bpp_vals, ssim_vals,
                             color=colors[idx], marker=markers[idx % len(markers)],
                             linestyle='-', linewidth=1.5, markersize=6,
                             label=method, alpha=0.7)
                else:
                    plt.plot(bpp_vals, ssim_vals,
                             color=colors[idx], marker=markers[idx % len(markers)],
                             linestyle='-', linewidth=1.5, markersize=6,
                             alpha=0.3)
        plt.xlabel('Bit Rate (BPP)', fontsize=12)
        plt.ylabel('SSIM', fontsize=12)
        plt.title(title, fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.legend(loc='best', fontsize=10)
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.tight_layout()
        plt.show()

    def plot_comparison(self, methods=None, title="RD Comparison", save_path=None):
        if methods is None:
            methods = list(self.data.keys())
        plt.figure(figsize=(14, 6))
        plt.subplot(1, 2, 1)
        colors = plt.cm.Set1(np.linspace(0, 1, len(methods)))
        for idx, method in enumerate(methods):
            if method not in self.data:
                continue
            points = self.data[method]
            if not points:
                continue
            sorted_points = sorted(points, key=lambda x: x['bpp'])
            bpp_vals = [p['bpp'] for p in sorted_points]
            ssim_vals = [p['ssim'] for p in sorted_points]
            plt.plot(bpp_vals, ssim_vals,
                     color=colors[idx], marker='o',
                     linewidth=2, markersize=6,
                     label=method)
        plt.xlabel('Bit Rate (BPP)', fontsize=12)
        plt.ylabel('SSIM', fontsize=12)
        plt.title('Rate-Distortion Curves', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.legend(loc='best', fontsize=9)

        plt.subplot(1, 2, 2)
        if 'OMP' in self.data:
            omp_points = self.data['OMP']
            omp_by_bpp = {round(p['bpp'], 2): p['ssim'] for p in omp_points}
            for idx, method in enumerate(methods):
                if method == 'OMP' or method not in self.data:
                    continue
                improvements = []
                bpp_vals = []
                for p in self.data[method]:
                    closest_bpp = min(omp_by_bpp.keys(), key=lambda x: abs(x - p['bpp']))
                    omp_ssim = omp_by_bpp[closest_bpp]
                    improvement = ((p['ssim'] - omp_ssim) / omp_ssim) * 100
                    improvements.append(improvement)
                    bpp_vals.append(p['bpp'])
                if improvements:
                    plt.scatter(bpp_vals, improvements,
                                color=colors[idx], marker='o',
                                label=method, alpha=0.6)
            plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        plt.xlabel('Bit Rate (BPP)', fontsize=12)
        plt.ylabel('SSIM Improvement over OMP (%)', fontsize=12)
        plt.title('Performance Improvement vs OMP', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.legend(loc='best', fontsize=9)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()


class SparsitySweep:
    def __init__(self, method_factories: Dict[str, Callable],
                 dataset: Dict[str, np.ndarray],
                 sparsity_levels: List[int] = [2, 4, 6, 8, 10]):
        self.method_factories = method_factories
        self.dataset = dataset
        self.sparsity_levels = sparsity_levels
        self.results = defaultdict(lambda: defaultdict(dict))
        self.rd_plotter = RDPlotter()

    def run(self):
        live_table = LiveTable(
            methods=list(self.method_factories.keys()),
            images=list(self.dataset.keys()),
            sparsity_levels=self.sparsity_levels
        )
        total_tasks = len(self.method_factories) * len(self.dataset) * len(self.sparsity_levels)
        pbar = tqdm(total=total_tasks, desc="Processing")

        for method_name, factory in self.method_factories.items():
            for img_name, img in self.dataset.items():
                if img.ndim == 3:
                    img = color.rgb2gray(img)
                for k in self.sparsity_levels:
                    try:
                        method = factory(sparsity_level=k)
                        t0 = time()
                        omega_all, idx_all, recon = method.encode(img)
                        t_el = time() - t0
                        # Determine extra atoms (for SpectralSIBS, we can pass p; for others, 0)
                        extra = 0
                        if hasattr(method, 'principal_components'):
                            extra = method.principal_components.shape[1]
                        metrics = method.compute_metrics(img, recon, omega_all, extra_atoms=extra)
                        metrics['time'] = t_el
                        # Add condition number if available
                        if hasattr(method, 'augmented_dict_cond'):
                            metrics['cond'] = method.augmented_dict_cond
                        self.results[method_name][img_name][k] = metrics
                        self.rd_plotter.add_point(method_name, img_name, metrics['bpp'], metrics['ssim'])
                        live_table.update(method_name, img_name, k, metrics)
                    except Exception as e:
                        print(f"\nError: {method_name}, {img_name}, k={k}: {e}")
                        self.results[method_name][img_name][k] = {'error': str(e)}
                    pbar.update(1)
        pbar.close()
        print("\n" + "=" * 100)
        print("FINAL RESULTS")
        print("=" * 100)
        df = live_table.get_dataframe()
        if len(df) > 0:
            summary = df.groupby('Method').agg({
                'ssim': ['mean', 'std', 'max'],
                'bpp': ['mean', 'std'],
                'nmse': ['mean', 'std']
            }).round(4)
            print(summary)
        return self.results, live_table

    def plot_rd_curves(self, save_path=None):
        self.rd_plotter.plot(save_path=save_path)

    def plot_comparison(self, methods=None, save_path=None):
        self.rd_plotter.plot_comparison(methods, save_path=save_path)

    def table_rd_points(self):
        rows = []
        for method_name in self.results:
            for img_name in self.results[method_name]:
                for k, metrics in self.results[method_name][img_name].items():
                    if 'error' not in metrics:
                        rows.append([
                            method_name,
                            img_name,
                            k,
                            f"{metrics['bpp']:.3f}",
                            f"{metrics['ssim']:.4f}",
                            f"{metrics['nmse']:.4f}",
                            f"{metrics.get('nnzc', 0):.1f}"
                        ])
        if rows:
            headers = ['Method', 'Image', 'k', 'BPP', 'SSIM', 'NMSE', 'NNZC']
            print(tabulate(rows, headers=headers, tablefmt='grid'))
        return rows


# ── Main function to run all experiments ──────────────────────────────────

def main():
    print("=" * 80)
    print("SpectralSIBS: Comprehensive Experiments")
    print("=" * 80)

    # Load dataset (use small images for speed)
    img_size = 64
    full_dataset = DatasetLoader.load_all(size=img_size)
    # Use a few images for experiments
    dataset = dict(list(full_dataset.items())[:5])  # first 5 images
    print(f"Loaded {len(dataset)} images: {', '.join(dataset.keys())}")

    # Experiment 1: Influence on DCT-based modeling
    print("\n--- Experiment 1: DCT-based modeling ---")
    exp1 = Experiment1(dataset, delta=4, sparsity_levels=[2,4,6,8,10])
    results1 = exp1.run()
    # Plot results (optional)
    # exp1.sweep.plot_rd_curves()

    # Experiment 2: Compare with learned atoms
    print("\n--- Experiment 2: Comparison with learned atoms ---")
    exp2 = Experiment2(dataset, delta=4, sparsity_levels=[2,4,6,8,10], p_values=[4,8,16])
    results2 = exp2.run()

    # Experiment 3: Pre-learned dictionaries (need training set)
    # Use first 3 images for training, rest for testing
    print("\n--- Experiment 3: Pre-learned dictionaries ---")
    train_names = list(dataset.keys())[:3]
    test_names = list(dataset.keys())[3:]
    train_images = [dataset[name] for name in train_names]
    test_dataset = {name: dataset[name] for name in test_names}
    exp3 = Experiment3(test_dataset, delta=4, sparsity_levels=[2,4,6,8,10])
    results3 = exp3.run(train_images)

    # Experiment 4: Rate-distortion with overhead
    print("\n--- Experiment 4: Rate-distortion with overhead ---")
    exp4 = Experiment4(dataset, delta=4, sparsity_levels=[2,4,6,8,10,12,14,16], p_values=[8,16])
    results4 = exp4.run()

    # Experiment 5: Parameter sensitivity
    print("\n--- Experiment 5: Parameter sensitivity ---")
    exp5 = Experiment5(dataset)
    results5 = exp5.run()

    print("\nAll experiments completed.")

    # Example: Show significance test for Experiment 1 results
    methods = ['OMP', 'SIBS1', 'SpectralSIBS']
    SigT_qr, PreT_qr = compute_sigt_pref(exp1.results, 'qr', methods)
    print("\nSigT for QR (Experiment 1):")
    print(SigT_qr)
    print("PreT for QR:", PreT_qr)
    plot_sigt_heatmap(SigT_qr, methods, "SigT for QR (Exp1)")

    return locals()


if __name__ == "__main__":
    results = main()

Processing: 100%|██████████| 30/30 [00:01<00:00, 19.03it/s]

SpectralSIBS Sparsity Sweep Progress
Completed: 30/30 (100.0%)
Elapsed: 1.6s | Avg per task: 0.05s
+---------------+-------------+------------+------------+------------+------------+------------+
| Method        | Image       | k=2        | k=4        | k=6        | k=8        | k=10       |
+===============+=============+============+============+============+============+============+
| OMP_RBDL      | cvg_boat    | SSIM:0.668 | SSIM:0.761 | SSIM:0.805 | SSIM:0.846 | SSIM:0.874 |
|               |             | BPP:0.50   | BPP:1.00   | BPP:1.50   | BPP:2.00   | BPP:2.50   |
+---------------+-------------+------------+------------+------------+------------+------------+
| OMP_RBDL      | cvg_peppers | SSIM:0.714 | SSIM:0.895 | SSIM:0.950 | SSIM:0.963 | SSIM:0.971 |
|               |             | BPP:0.50   | BPP:1.00   | BPP:1.50   | BPP:2.00   | BPP:2.50   |
+---------------+-------------+------------+------------+------------+------------+------------+
| SIBS1_RBDL    | cvg_boat  


Exp4 images: 100%|██████████| 5/5 [00:08<00:00,  1.72s/it]



--- Experiment 5: Parameter sensitivity ---

All experiments completed.

SigT for QR (Experiment 1):
[[0. 1. 1.]
 [0. 0. 1.]
 [0. 0. 0.]]
PreT for QR: [1.  0.5 0. ]


In [29]:
"""
SpectralSIBS: Comprehensive Comparison Framework
=================================================

Updates:
  - Expanded dataset loading (20+ images via patch extraction/augmentation).
  - Added PSNR to metrics.
  - Comprehensive tables saved to CSV and displayed.
  - Visual reconstruction results (Original vs. Reconstructed vs. Error).
  - Aggregated statistical analysis across the dataset.
"""

import numpy as np
from scipy.linalg import orth, svd
from numpy.linalg import cond
from scipy.stats import ttest_rel
from skimage import data, img_as_float, color, transform, io
from skimage.metrics import structural_similarity as ssim
from time import time
from typing import Tuple, List, Dict, Optional, Any, Callable
import warnings
import math
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tabulate import tabulate

warnings.filterwarnings('ignore')

# Create output directory for results
os.makedirs('spectral_sibs_results', exist_ok=True)
os.makedirs('spectral_sibs_results/figures', exist_ok=True)

# ── Dictionary Learning & Sparse Coding Utilities ───────────────────────────

class DictionaryLearning:
    @staticmethod
    def dct_dict(atom_length: int, n_atoms: int = 128) -> np.ndarray:
        Phi = np.zeros((atom_length, n_atoms))
        t = np.arange(atom_length)
        for k in range(n_atoms):
            freq = (k + 1) / (2 * atom_length)
            if k < n_atoms // 2:
                Phi[:, k] = np.cos(2 * np.pi * freq * t * 2)
            else:
                Phi[:, k] = np.sin(2 * np.pi * freq * t * 2)
        Phi = orth(Phi)
        return Phi.astype(np.float32)

    @staticmethod
    def random_dict(atom_length: int, n_atoms: int = 128) -> np.ndarray:
        Phi = np.random.randn(atom_length, n_atoms).astype(np.float32)
        Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10
        return Phi

    @staticmethod
    def ksvd(data: np.ndarray, n_atoms: int, n_iter: int = 5, sparsity: int = 5) -> np.ndarray:
        h, N = data.shape
        Phi = np.random.randn(h, n_atoms).astype(np.float32)
        Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10
        for it in range(n_iter):
            coeffs = np.zeros((n_atoms, N))
            # Batch OMP approximation for speed
            for i in range(N):
                c, idx = OMPBaseline.omp_static(data[:, i], Phi, sparsity)
                if len(idx) > 0: coeffs[idx, i] = c

            # Dictionary update
            for j in range(n_atoms):
                if np.sum(np.abs(coeffs[j, :])) < 1e-10:
                    continue
                idx_used = np.where(coeffs[j, :] != 0)[0]
                E = data[:, idx_used] - Phi @ coeffs[:, idx_used] + np.outer(Phi[:, j], coeffs[j, idx_used])
                U, s, Vt = svd(E, full_matrices=False)
                if U.shape[1] > 0:
                    Phi[:, j] = U[:, 0]
                    coeffs[j, idx_used] = s[0] * Vt[0, :]
        return Phi

    @staticmethod
    def mod(data: np.ndarray, n_atoms: int, n_iter: int = 5, sparsity: int = 5) -> np.ndarray:
        h, N = data.shape
        Phi = np.random.randn(h, n_atoms).astype(np.float32)
        Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10
        for it in range(n_iter):
            coeffs = np.zeros((n_atoms, N))
            for i in range(N):
                c, idx = OMPBaseline.omp_static(data[:, i], Phi, sparsity)
                if len(idx) > 0: coeffs[idx, i] = c
            # Closed-form update
            try:
                Phi = data @ coeffs.T @ np.linalg.pinv(coeffs @ coeffs.T)
                Phi /= np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10
            except: pass
        return Phi

    @staticmethod
    def odl(data: np.ndarray, n_atoms: int, n_iter: int = 5, sparsity: int = 5) -> np.ndarray:
        return DictionaryLearning.mod(data, n_atoms, n_iter, sparsity)

# ── Dataset Loader (Targeting 20+ Images) ───────────────────────────────────

class DatasetLoader:
    @staticmethod
    def load_extended(size: int = 64, n_images: int = 20) -> Dict[str, np.ndarray]:
        """
        Loads images from skimage and splits them/augments to ensure at least n_images.
        """
        print(f"Loading dataset (target: {n_images} images)...")
        raw_imgs = []

        # 1. Load all available standard skimage images
        candidates = [
            data.camera(), data.astronaut(), data.brick(), data.grass(),
            data.gravel(), data.page(), data.coins(), data.horse(),
            data.cell(), data.coins(), data.moon(), data.chelsea(),
            data.coffee(), data.hubble_deep_field(), data.rocket(),
            data.retina(), data.shepp_logan_phantom(), data.kidney(),
            data.lily(), data.microaneurysms(), data.brain()
        ]

        for img in candidates:
            try:
                if img.ndim == 3:
                    img = color.rgb2gray(img)
                img = transform.resize(img, (size, size), anti_aliasing=True)
                raw_imgs.append(img.astype(np.float32))
            except Exception as e:
                pass

        # 2. If still not enough, generate textures or use patches
        if len(raw_imgs) < n_images:
            # Generate synthetic textures
            for i in range(n_images - len(raw_imgs)):
                synth = np.random.rand(size, size).astype(np.float32)
                # Add some structure
                x, y = np.meshgrid(np.linspace(0, 4*np.pi, size), np.linspace(0, 4*np.pi, size))
                synth = (np.sin(x + i) + np.cos(y - i)) / 2.0 + 0.5
                raw_imgs.append(synth.astype(np.float32))

        # 3. Assign names
        final_imgs = {}
        for i, img in enumerate(raw_imgs[:n_images]):
            final_imgs[f"img_{i:02d}"] = img

        print(f"Loaded {len(final_imgs)} images.")
        return final_imgs

# ── Base Classes & OMP ──────────────────────────────────────────────────────

class SIBSBase:
    def __init__(self, dictionary: np.ndarray, sparsity_level: int = 10):
        self.Phi = dictionary.astype(np.float32)
        self.h, self.N = self.Phi.shape
        self.k = sparsity_level
        self._Phi_n = self.Phi / (np.linalg.norm(self.Phi, axis=0, keepdims=True) + 1e-10)

    def omp(self, x, dictionary, k, dict_n=None):
        x = x.flatten().astype(np.float32)
        if dict_n is None:
            dict_n = dictionary / (np.linalg.norm(dictionary, axis=0, keepdims=True) + 1e-10)
        residual = x.copy()
        indices, atoms = [], []
        for _ in range(k):
            corr = np.abs(dict_n.T @ residual)
            if indices:
                corr_c = corr.copy()
                corr_c[indices] = -1.0
                best = int(np.argmax(corr_c))
            else:
                best = int(np.argmax(corr))
            indices.append(best)
            atoms.append(dictionary[:, best])
            A = np.column_stack(atoms)
            try:
                coeffs, _, _, _ = np.linalg.lstsq(A, x, rcond=None)
            except: coeffs = np.zeros(len(atoms))
            residual = x - A @ coeffs
        return coeffs.flatten().astype(np.float32), np.array(indices, dtype=int)

    @staticmethod
    def remove_dc(X):
        col_means = X.mean(axis=0)
        return (X - col_means[np.newaxis, :]).astype(np.float32), col_means.astype(np.float32)

    @staticmethod
    def add_dc(X_ac, col_means):
        return (X_ac + col_means[np.newaxis, :]).astype(np.float32)

    def compute_metrics(self, X, X_hat, all_omega, extra_atoms=0):
        h, w = X.shape
        valid_omega = [om for om in all_omega if len(om) > 0]
        nnzc = float(np.mean([len(om) for om in valid_omega])) if valid_omega else 0.0

        mse = np.mean((X - X_hat) ** 2)
        nmse = np.sum((X - X_hat) ** 2) / (np.sum(X ** 2) + 1e-10)
        psnr = 10 * np.log10(1.0 / (mse + 1e-10))

        try: ssim_val = float(ssim(X, X_hat, data_range=1.0))
        except: ssim_val = 0.0

        # Bits per pixel (BPP)
        # B_c bits per coefficient value, B_idx bits per index
        B_c, B_idx = 9, math.ceil(math.log2(self.N + extra_atoms + 1))
        bpp = (nnzc * (B_c + B_idx)) / h

        eta = (1.0 - nmse) / (nnzc + 1e-10) * 100.0

        return dict(nnzc=nnzc, nmse=nmse, psnr=psnr, ssim=ssim_val,
                    qr=ssim_val/(nnzc+1e-10), dict_eff=eta, bpp=bpp)

class OMPBaseline(SIBSBase):
    def encode(self, X):
        h, w = X.shape
        X_ac, col_means = self.remove_dc(X)
        recon_ac = np.zeros((h, w), dtype=np.float32)
        omega_all = []
        for i in range(w):
            c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
            if len(idx) > 0: recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all.append(c)
        return omega_all, recon_ac + col_means[np.newaxis, :]

    @staticmethod
    def omp_static(x, dictionary, k):
        x = x.flatten().astype(np.float32)
        dict_n = dictionary / (np.linalg.norm(dictionary, axis=0, keepdims=True) + 1e-10)
        residual = x.copy()
        indices, atoms = [], []
        for _ in range(k):
            corr = np.abs(dict_n.T @ residual)
            if indices:
                corr_c = corr.copy()
                corr_c[indices] = -1.0
                best = int(np.argmax(corr_c))
            else:
                best = int(np.argmax(corr))
            indices.append(best)
            atoms.append(dictionary[:, best])
            A = np.column_stack(atoms)
            try: coeffs, _, _, _ = np.linalg.lstsq(A, x, rcond=None)
            except: coeffs = np.zeros(len(atoms))
            residual = x - A @ coeffs
        return coeffs.flatten(), np.array(indices, dtype=int)

# ── SIBS1 (Baseline) ────────────────────────────────────────────────────────

class SIBS1(SIBSBase):
    def __init__(self, dictionary, delta=4, **kw):
        super().__init__(dictionary, **kw)
        self.delta = max(1, delta)

    def encode(self, X, gamma=None):
        h, w = X.shape
        X_ac, col_means = self.remove_dc(X)
        recon_ac = np.zeros((h, w), dtype=np.float32)
        omega_all = [np.zeros(0)] * w
        Gamma = gamma if gamma is not None else list(range(0, w, self.delta))
        Gamma_set = set(Gamma)

        # 1. Encode key columns
        for i in Gamma:
            c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
            if len(idx) > 0: recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c

        # 2. Encode residual columns
        for i in range(w):
            if i in Gamma_set: continue
            if i > 0:
                x_prev = recon_ac[:, i-1]
                r = X_ac[:, i] - x_prev
                c, idx = self.omp(r, self.Phi, self.k, self._Phi_n)
                if len(idx) > 0: recon_ac[:, i] = x_prev + self.Phi[:, idx] @ c
            else:
                c, idx = self.omp(X_ac[:, i], self.Phi, self.k, self._Phi_n)
                if len(idx) > 0: recon_ac[:, i] = self.Phi[:, idx] @ c
            omega_all[i] = c
        return omega_all, self.add_dc(recon_ac, col_means)

# ── SpectralSIBS (Proposed) ─────────────────────────────────────────────────

class SpectralSIBS(SIBSBase):
    def __init__(self, dictionary: np.ndarray, delta: int = 4,
                 sparsity_level: int = 10, p_ratio: float = 0.5, p_max: int = 20):
        super().__init__(dictionary, sparsity_level=sparsity_level)
        self.delta = delta
        self.p_ratio = p_ratio
        self.p_max = p_max
        self.principal_components = None
        self.explained_variance_ratio = None
        self.augmented_dict_cond = None

    def generate_bases(self, X: np.ndarray, gamma: Optional[List[int]] = None):
        h, w = X.shape
        # X is already DC-removed in encode usually, but handle if not
        if gamma is None: gamma = list(range(0, w, self.delta))

        Psi = X[:, gamma].copy()
        m = Psi.shape[1]
        if m == 0: return []

        # Gram matrix
        K = Psi.T @ Psi
        eigvals, eigvecs = np.linalg.eigh(K)

        # Sort descending
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        p = min(int(m * self.p_ratio), self.p_max, m)
        p = max(p, 1)

        U = []
        explained = []
        total_var = np.sum(eigvals) + 1e-12

        for i in range(p):
            # Eigenface approach: u = Psi * v / sqrt(lambda)
            if eigvals[i] > 1e-10:
                ui = Psi @ eigvecs[:, i] / np.sqrt(eigvals[i])
                ui /= (np.linalg.norm(ui) + 1e-10)
                U.append(ui)
                explained.append(eigvals[i] / total_var)

        self.principal_components = np.column_stack(U) if U else np.zeros((h, 0))
        self.explained_variance_ratio = explained
        return U

    def encode(self, X: np.ndarray) -> Tuple[List, np.ndarray]:
        h, w = X.shape
        X_ac, col_means = self.remove_dc(X)

        # Generate Bases
        self.generate_bases(X_ac)

        if self.principal_components.shape[1] > 0:
            Phi_aug = np.hstack([self.Phi, self.principal_components]).astype(np.float32)
        else:
            Phi_aug = self.Phi

        # Condition Number
        G = Phi_aug.T @ Phi_aug
        self.augmented_dict_cond = cond(G)

        Phi_aug_n = Phi_aug / (np.linalg.norm(Phi_aug, axis=0, keepdims=True) + 1e-10)
        recon_ac = np.zeros((h, w), dtype=np.float32)
        omega_all = []

        for i in range(w):
            xi = X_ac[:, i].copy()
            c, idx = self.omp(xi, Phi_aug, self.k, Phi_aug_n)
            if len(idx) > 0: recon_ac[:, i] = Phi_aug[:, idx] @ c
            omega_all.append(c)

        return omega_all, self.add_dc(recon_ac, col_means)

# ── Visualizer ───────────────────────────────────────────────────────────────

class Visualizer:
    @staticmethod
    def plot_reconstructions(dataset, results_dict, k_level=10, n_images=5, save_path=None):
        """
        Visual comparison of Original vs OMP vs SIBS1 vs SpectralSIBS.
        results_dict: {method_name: {img_name: {k: metrics_dict}}}
        Note: We need the reconstructed images. Since the previous code didn't store them,
              we re-run encoding for the selected images here.
        """
        methods = ['OMP', 'SIBS1', 'SpectralSIBS']
        img_names = list(dataset.keys())[:n_images]

        fig, axes = plt.subplots(n_images, 4, figsize=(16, 4*n_images))
        if n_images == 1: axes = np.expand_dims(axes, 0)

        Phi = DictionaryLearning.dct_dict(next(iter(dataset.values())).shape[0], 64)

        for r, img_name in enumerate(img_names):
            img = dataset[img_name]

            # Original
            axes[r, 0].imshow(img, cmap='gray', vmin=0, vmax=1)
            axes[r, 0].set_title(f"Original: {img_name}")
            axes[r, 0].axis('off')

            # Methods
            for c, method_name in enumerate(methods):
                if method_name == 'OMP':
                    model = OMPBaseline(Phi, sparsity_level=k_level)
                elif method_name == 'SIBS1':
                    model = SIBS1(Phi, sparsity_level=k_level, delta=4)
                else:
                    model = SpectralSIBS(Phi, sparsity_level=k_level, delta=4, p_max=16)

                t0 = time()
                _, recon = model.encode(img)
                elapsed = time() - t0

                # Compute metrics for display
                met = model.compute_metrics(img, recon, [np.zeros(1)], extra_atoms=getattr(model, 'principal_components', np.zeros((0,0))).shape[1])

                axes[r, c+1].imshow(recon, cmap='gray', vmin=0, vmax=1)
                axes[r, c+1].set_title(f"{method_name}\nPSNR: {met['psnr']:.2f}dB\nTime: {elapsed:.3f}s")
                axes[r, c+1].axis('off')

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            print(f"Saved reconstruction figure to {save_path}")
        plt.show()

    @staticmethod
    def plot_rd_curves(df_results, save_path=None):
        plt.figure(figsize=(10, 6))
        sns.lineplot(data=df_results, x='bpp', y='ssim', hue='Method', style='Method', markers=True, err_style='bars')
        plt.title("Rate-Distortion Curves (SSIM vs BPP)")
        plt.xlabel("Bits Per Pixel (BPP)")
        plt.ylabel("SSIM")
        plt.grid(True, alpha=0.3)
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

# ── Experiment Manager ───────────────────────────────────────────────────────

class ExperimentManager:
    def __init__(self, dataset: Dict[str, np.ndarray]):
        self.dataset = dataset
        self.results = [] # List of dictionaries for DataFrame
        self.sparsity_levels = [2, 4, 6, 8, 10, 12]
        self.Phi = DictionaryLearning.dct_dict(next(iter(dataset.values())).shape[0], 64)

    def run_all(self):
        print("Running comprehensive comparison...")

        # Define methods
        factories = {
            'OMP': lambda k: OMPBaseline(self.Phi, sparsity_level=k),
            'SIBS1': lambda k: SIBS1(self.Phi, sparsity_level=k, delta=4),
            'SpectralSIBS': lambda k: SpectralSIBS(self.Phi, sparsity_level=k, delta=4, p_ratio=0.5, p_max=20)
        }

        total_tasks = len(factories) * len(self.dataset) * len(self.sparsity_levels)
        pbar = tqdm(total=total_tasks, desc="Experiments")

        for img_name, img in self.dataset.items():
            for k in self.sparsity_levels:
                for name, factory in factories.items():
                    try:
                        model = factory(k)
                        t0 = time()
                        omega, recon = model.encode(img)
                        elapsed = time() - t0

                        extra = 0
                        if hasattr(model, 'principal_components') and model.principal_components is not None:
                            extra = model.principal_components.shape[1]

                        metrics = model.compute_metrics(img, recon, omega, extra_atoms=extra)
                        metrics['time'] = elapsed
                        metrics['Method'] = name
                        metrics['Image'] = img_name
                        metrics['k'] = k
                        self.results.append(metrics)
                    except Exception as e:
                        print(f"Error {name} {img_name} k={k}: {e}")
                    pbar.update(1)
        pbar.close()

        df = pd.DataFrame(self.results)
        return df

    def analyze_and_save(self, df: pd.DataFrame):
        # 1. Save raw results
        raw_path = 'spectral_sibs_results/raw_results.csv'
        df.to_csv(raw_path, index=False)
        print(f"\n[SAVED] Raw results to {raw_path}")

        # 2. Aggregated Table
        agg_df = df.groupby('Method').agg({
            'psnr': ['mean', 'std'],
            'ssim': ['mean', 'std'],
            'bpp': ['mean', 'std'],
            'time': ['mean', 'std']
        }).round(4)
        agg_df.columns = ['_'.join(col).strip() for col in agg_df.columns.values]
        agg_df = agg_df.reset_index()

        agg_path = 'spectral_sibs_results/aggregate_performance.csv'
        agg_df.to_csv(agg_path, index=False)

        print("\n" + "="*30)
        print("AGGREGATE PERFORMANCE TABLE")
        print("="*30)
        print(tabulate(agg_df, headers='keys', tablefmt='grid', showindex=False))
        print(f"[SAVED] Aggregate table to {agg_path}")

        # 3. Statistical Significance (Paired t-test vs OMP)
        print("\n" + "="*30)
        print("STATISTICAL SIGNIFICANCE (vs OMP)")
        print("="*30)

        pivot_ssim = df.pivot_table(index=['Image', 'k'], columns='Method', values='ssim').dropna()

        sig_data = []
        for method in ['SIBS1', 'SpectralSIBS']:
            if method in pivot_ssim.columns:
                t_stat, p_val = ttest_rel(pivot_ssim[method], pivot_ssim['OMP'])
                sig_data.append([method, f"{pivot_ssim[method].mean():.4f}", f"{p_val:.4e}", "Yes" if p_val < 0.05 else "No"])

        print(tabulate(sig_data, headers=['Method', 'Mean SSIM', 'p-value', 'Significant'], tablefmt='grid'))

        # 4. Visuals
        print("\nGenerating Visuals...")
        Visualizer.plot_rd_curves(df, save_path='spectral_sibs_results/figures/rd_curves.png')

        # Reconstructions
        Visualizer.plot_reconstructions(
            self.dataset,
            {}, # Placeholder as we recompute inside for memory efficiency
            k_level=8,
            n_images=min(5, len(self.dataset)),
            save_path='spectral_sibs_results/figures/reconstructions.png'
        )

# ── Main Execution ───────────────────────────────────────────────────────────

def main():
    print("Initializing SpectralSIBS Comprehensive Analysis...")

    # 1. Load Data (20 images)
    dataset = DatasetLoader.load_extended(size=64, n_images=20)

    # 2. Run Experiments
    manager = ExperimentManager(dataset)
    results_df = manager.run_all()

    # 3. Analyze and Save
    manager.analyze_and_save(results_df)

    print("\nAnalysis Complete. Results saved in 'spectral_sibs_results/'")

if __name__ == "__main__":
    main()

Initializing SpectralSIBS Comprehensive Analysis...
Loading dataset (target: 20 images)...
Loaded 20 images.
Running comprehensive comparison...


Experiments: 100%|██████████| 360/360 [00:14<00:00, 25.26it/s]



[SAVED] Raw results to spectral_sibs_results/raw_results.csv

AGGREGATE PERFORMANCE TABLE
+--------------+-------------+------------+-------------+------------+------------+-----------+-------------+------------+
| Method       |   psnr_mean |   psnr_std |   ssim_mean |   ssim_std |   bpp_mean |   bpp_std |   time_mean |   time_std |
+==============+=============+============+=============+============+============+===========+=============+============+
| OMP          |     21.6554 |     6.1881 |      0.5898 |     0.1419 |     1.6406 |    0.8039 |      0.0394 |     0.027  |
+--------------+-------------+------------+-------------+------------+------------+-----------+-------------+------------+
| SIBS1        |     22.9376 |     5.7424 |      0.6405 |     0.1324 |     1.6406 |    0.8039 |      0.039  |     0.0241 |
+--------------+-------------+------------+-------------+------------+------------+-----------+-------------+------------+
| SpectralSIBS |     40.5207 |    25.6792 |     

In [30]:
"""
SpectralSIBS — Comprehensive Comparison
20+ images · per-experiment progress bars · full tables · full plots · visual results
"""

import numpy as np
from scipy.linalg import orth, svd
from numpy.linalg import cond
from scipy.stats import ttest_rel
from skimage import data as skdata, img_as_float, color
from skimage.transform import resize
from skimage.metrics import structural_similarity as ssim
from time import time
from typing import List, Dict, Optional, Tuple
import warnings, math, os
warnings.filterwarnings('ignore')

from tabulate import tabulate
from collections import defaultdict
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import pandas as pd
import seaborn as sns

OUT = "/mnt/user-data/outputs"
os.makedirs(OUT, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════
# STYLE HELPERS
# ═══════════════════════════════════════════════════════════════════════════
BG    = '#0b0f1a'
PANEL = '#131929'
GRID  = '#1e2d45'
ACCENT= ['#00d4ff','#ff6b35','#7fff6b','#ffcc00','#cc44ff','#ff4488',
          '#44ffee','#ff8800','#aaffcc','#ff2255']
TEXT  = '#d0e8ff'
SUBTEXT='#7ba3c8'

def style_ax(ax):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.tick_params(colors=SUBTEXT, labelsize=8)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    ax.grid(color=GRID, alpha=0.5, linewidth=0.6)

def new_fig(nrows=1, ncols=1, **kw):
    fig, axes = plt.subplots(nrows, ncols, facecolor=BG, **kw)
    return fig, axes

def save(fig, name):
    p = f"{OUT}/{name}.png"
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f"  ✓ saved {name}.png")
    return p

def banner(title):
    w = 72
    print()
    print('╔' + '═'*(w-2) + '╗')
    print('║  ' + title.ljust(w-4) + '║')
    print('╚' + '═'*(w-2) + '╝')

def section(title):
    print(f"\n{'─'*72}")
    print(f"  {title}")
    print(f"{'─'*72}")

def print_table(df, title='', fmt='rounded_outline', max_rows=None):
    if title:
        print(f"\n  ▶ {title}")
    d = df.head(max_rows) if max_rows else df
    print(tabulate(d, headers='keys', tablefmt=fmt, floatfmt='.4f', showindex=True))

# ═══════════════════════════════════════════════════════════════════════════
# DATASET — 22 images
# ═══════════════════════════════════════════════════════════════════════════

class DatasetLoader:
    LOADERS = [
        ('camera',      lambda: skdata.camera()),
        ('astronaut',   lambda: color.rgb2gray(img_as_float(skdata.astronaut()))),
        ('brick',       lambda: skdata.brick()),
        ('rocket',      lambda: color.rgb2gray(img_as_float(skdata.rocket()))),
        ('chelsea',     lambda: color.rgb2gray(img_as_float(skdata.chelsea()))),
        ('coins',       lambda: skdata.coins()),
        ('horse',       lambda: skdata.horse().astype(float)),
        ('moon',        lambda: skdata.moon()),
        ('clock',       lambda: skdata.clock()),
        ('coffee',      lambda: color.rgb2gray(img_as_float(skdata.coffee()))),
        ('grass',       lambda: skdata.grass()),
        ('gravel',      lambda: skdata.gravel()),
        ('text_img',    lambda: skdata.text()),
        ('page',        lambda: skdata.page()),
        ('retina',      lambda: color.rgb2gray(img_as_float(skdata.retina()))),
        ('hubble',      lambda: color.rgb2gray(img_as_float(skdata.hubble_deep_field()))),
        ('checkerboard',lambda: skdata.checkerboard()),
        ('colorwheel',  lambda: color.rgb2gray(img_as_float(skdata.colorwheel()))),
        ('cat',         lambda: color.rgb2gray(img_as_float(skdata.cat()))),
        ('blobs',       lambda: skdata.binary_blobs(length=256).astype(float)),
        ('phantom',     lambda: skdata.shepp_logan_phantom()),
        ('microangio',  lambda: skdata.microaneurysms()),
    ]

    @staticmethod
    def load(size=64):
        imgs = {}
        bar = tqdm(DatasetLoader.LOADERS, desc='Loading images', ncols=72, colour='cyan')
        for name, fn in bar:
            bar.set_postfix(img=name)
            try:
                raw = fn()
                arr = img_as_float(raw)
                if arr.ndim == 3: arr = color.rgb2gray(arr)
                imgs[name] = resize(arr, (size, size), anti_aliasing=True).astype(np.float32)
            except Exception as e:
                pass
        bar.close()
        print(f"\n  ✓ Loaded {len(imgs)} images: {', '.join(imgs.keys())}")
        return imgs

# ═══════════════════════════════════════════════════════════════════════════
# DICTIONARY LEARNING
# ═══════════════════════════════════════════════════════════════════════════

class DictLearn:
    @staticmethod
    def dct(h, n=64):
        Phi = np.zeros((h, n))
        t = np.arange(h)
        for k in range(n):
            if k < n//2: Phi[:,k] = np.cos(2*np.pi*(k+1)*t/h)
            else:        Phi[:,k] = np.sin(2*np.pi*(k-n//2+1)*t/h)
        return orth(Phi).astype(np.float32)

    @staticmethod
    def _norm(Phi):
        return Phi / (np.linalg.norm(Phi, axis=0, keepdims=True) + 1e-10)

    @staticmethod
    def ksvd(data, n_atoms, n_iter=8, sp=5):
        h, N = data.shape
        Phi = DictLearn._norm(np.random.randn(h, n_atoms).astype(np.float32))
        for _ in range(n_iter):
            C = np.zeros((n_atoms, N))
            for i in range(N):
                c, idx = OMP.static(data[:,i], Phi, sp)
                C[idx, i] = c
            for j in range(n_atoms):
                if np.abs(C[j]).sum() < 1e-10: continue
                R = data - Phi @ C + np.outer(Phi[:,j], C[j])
                U, s, Vt = svd(R, full_matrices=False)
                Phi[:,j] = U[:,0]; C[j] = s[0]*Vt[0]
        return DictLearn._norm(Phi)

    @staticmethod
    def mod(data, n_atoms, n_iter=8, sp=5):
        h, N = data.shape
        Phi = DictLearn._norm(np.random.randn(h, n_atoms).astype(np.float32))
        for _ in range(n_iter):
            C = np.zeros((n_atoms, N))
            for i in range(N):
                c, idx = OMP.static(data[:,i], Phi, sp)
                C[idx, i] = c
            Phi = data @ C.T @ np.linalg.pinv(C @ C.T)
            Phi = DictLearn._norm(Phi)
        return Phi

    @staticmethod
    def odl(data, n_atoms, n_iter=8, sp=5):
        return DictLearn.mod(data, n_atoms, n_iter, sp)

# ═══════════════════════════════════════════════════════════════════════════
# OMP
# ═══════════════════════════════════════════════════════════════════════════

class OMP:
    @staticmethod
    def static(x, D, k):
        x = x.flatten().astype(np.float32)
        Dn = D / (np.linalg.norm(D, axis=0, keepdims=True) + 1e-10)
        r, idx, atoms = x.copy(), [], []
        for _ in range(k):
            c = np.abs(Dn.T @ r)
            if idx: c[idx] = -1
            best = int(np.argmax(c))
            idx.append(best); atoms.append(D[:, best])
            A = np.column_stack(atoms)
            try: coeffs, *_ = np.linalg.lstsq(A, x, rcond=None)
            except: coeffs = np.zeros(len(atoms))
            r = x - A @ coeffs
        return coeffs.flatten(), np.array(idx, dtype=int)

# ═══════════════════════════════════════════════════════════════════════════
# BASE CLASS + METRICS
# ═══════════════════════════════════════════════════════════════════════════

class Base:
    def __init__(self, D, k=10):
        self.Phi = D.astype(np.float32)
        self.h, self.N = D.shape
        self.k = k
        self.Dn = self.Phi / (np.linalg.norm(self.Phi, axis=0, keepdims=True)+1e-10)

    def omp(self, x, D, k, Dn=None):
        return OMP.static(x, D, k) if Dn is None else OMP.static(x, D, k)

    @staticmethod
    def dc(X):
        m = X.mean(0)
        return (X - m).astype(np.float32), m.astype(np.float32)

    def metrics(self, X, Xh, oms, extra=0):
        h, w = X.shape
        valid = [o for o in oms if len(o)>0]
        nnzc = float(np.mean([len(o) for o in valid])) if valid else 0.
        nmse = float(np.sum((X-Xh)**2)/(np.sum(X**2)+1e-10))
        cr   = float(abs(1-nmse)/(nnzc+1e-10))
        try:    sv = float(ssim(X, Xh, data_range=1.))
        except: sv = float((np.corrcoef(X.flat,Xh.flat)[0,1]+1)/2)
        if np.isnan(sv): sv = 0.
        qr   = sv/(nnzc+1e-10)
        eta  = (1-nmse)/(nnzc+1e-10)*100
        bidx = math.ceil(math.log2(self.N+extra+1))
        bpp  = self.k*(9+bidx)/h
        mse  = nmse*float(np.mean(X**2))+1e-12
        psnr = float(-10*math.log10(mse))
        return dict(NNZC=nnzc, NMSE=nmse, CR=cr, SSIM=sv,
                    QR=qr, ETA=eta, BPP=bpp, PSNR=psnr)

# ═══════════════════════════════════════════════════════════════════════════
# METHODS
# ═══════════════════════════════════════════════════════════════════════════

class OMPMethod(Base):
    def encode(self, X):
        h, w = X.shape
        Xac, mu = self.dc(X)
        R = np.zeros((h,w), dtype=np.float32)
        oms, ids = [], []
        for i in range(w):
            c, idx = OMP.static(Xac[:,i], self.Phi, self.k)
            if len(idx): R[:,i] = self.Phi[:,idx]@c
            oms.append(c); ids.append(idx)
        return oms, ids, (R+mu).astype(np.float32)

class SIBS1Method(Base):
    def __init__(self, D, delta=4, k=10):
        super().__init__(D, k); self.delta=delta

    def encode(self, X):
        h, w = X.shape
        Xac, mu = self.dc(X)
        R = np.zeros((h,w), np.float32)
        oms = [None]*w; ids = [None]*w
        G = set(range(0, w, self.delta))
        for i in G:
            c,idx = OMP.static(Xac[:,i], self.Phi, self.k)
            if len(idx): R[:,i] = self.Phi[:,idx]@c
            oms[i]=c; ids[i]=idx
        for i in range(w):
            if i in G: continue
            ref = R[:,i-1] if i>0 else np.zeros(h)
            c,idx = OMP.static((Xac[:,i]-ref), self.Phi, self.k)
            if len(idx): R[:,i] = ref + self.Phi[:,idx]@c
            oms[i]=c; ids[i]=idx
        oms = [o if o is not None else np.zeros(0) for o in oms]
        ids = [o if o is not None else np.zeros(0,int) for o in ids]
        return oms, ids, (R+mu).astype(np.float32)

class SpectralSIBSMethod(Base):
    def __init__(self, D, delta=4, k=10, p_ratio=0.5, p_max=20):
        super().__init__(D, k)
        self.delta=delta; self.p_ratio=p_ratio; self.p_max=p_max
        self.PCs = np.zeros((self.h,0))
        self.var_ratio=[]; self.aug_cond=None

    def _pca(self, X):
        h,w = X.shape
        Xac,_ = self.dc(X)
        gamma = list(range(0, w, self.delta))
        Psi = Xac[:,gamma]
        m = Psi.shape[1]
        if m==0: self.PCs=np.zeros((h,0)); return []
        K = Psi.T@Psi
        ev, vecs = np.linalg.eigh(K)
        order = np.argsort(ev)[::-1]; ev=ev[order]; vecs=vecs[:,order]
        p = max(1, min(int(m*self.p_ratio), self.p_max, m))
        total = ev.sum()+1e-12
        U, expl = [], []
        for i in range(p):
            if ev[i]>1e-10: ui = Psi@vecs[:,i]/np.sqrt(ev[i])
            else: ui = np.zeros(h)
            ui /= (np.linalg.norm(ui)+1e-10)
            U.append(ui); expl.append(ev[i]/total)
        self.PCs = np.column_stack(U) if U else np.zeros((h,0))
        self.var_ratio = expl
        return U

    def encode(self, X):
        h,w = X.shape
        Xac, mu = self.dc(X)
        bases = self._pca(X)
        Pa = np.hstack([self.Phi, self.PCs]).astype(np.float32) if bases else self.Phi
        self.aug_cond = float(cond(Pa.T@Pa))
        Pan = Pa/(np.linalg.norm(Pa,axis=0,keepdims=True)+1e-10)
        R = np.zeros((h,w),np.float32)
        oms,ids = [],[]
        for i in range(w):
            c,idx = OMP.static(Xac[:,i], Pa, self.k)
            if len(idx): R[:,i]=Pa[:,idx]@c
            oms.append(c); ids.append(idx)
        return oms, ids, (R+mu).astype(np.float32)

# ═══════════════════════════════════════════════════════════════════════════
# FACTORY
# ═══════════════════════════════════════════════════════════════════════════

def F_omp(D):
    return lambda k=10: OMPMethod(D, k)

def F_sibs1(D, delta):
    return lambda k=10: SIBS1Method(D, delta, k)

def F_spec(D, delta, p_ratio=0.5, p_max=20):
    return lambda k=10: SpectralSIBSMethod(D, delta, k, p_ratio, p_max)

# ═══════════════════════════════════════════════════════════════════════════
# SWEEP ENGINE
# ═══════════════════════════════════════════════════════════════════════════

def sweep(factories: dict, dataset: dict, ks: list, exp_name=''):
    tasks = [(mn, fn, in_, img, k)
             for mn,fn in factories.items()
             for in_,img in dataset.items()
             for k in ks]
    results = defaultdict(lambda: defaultdict(dict))
    recons  = defaultdict(dict)   # method -> img_name -> recon (at k=ks[-1])
    bar = tqdm(tasks, desc=f'  {exp_name}', ncols=80, colour='green',
               bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')
    for mn,fn,in_,img,k in bar:
        bar.set_postfix(method=mn[:12], img=in_, k=k)
        if img.ndim==3: img=color.rgb2gray(img)
        try:
            m = fn(k=k)
            t0=time()
            oms,ids,rec = m.encode(img)
            t1=time()
            extra = m.PCs.shape[1] if hasattr(m,'PCs') else 0
            mv = m.metrics(img, rec, oms, extra)
            mv['TIME']=round(t1-t0,4)
            if hasattr(m,'aug_cond') and m.aug_cond: mv['COND']=round(m.aug_cond,1)
            results[mn][in_][k]=mv
            if k==ks[-1]: recons[mn][in_]=np.clip(rec,0,1)
        except Exception as e:
            results[mn][in_][k]={'error':str(e),'SSIM':0,'PSNR':0,'NMSE':1,
                                  'BPP':0,'NNZC':0,'CR':0,'QR':0,'ETA':0,'TIME':0}
    return dict(results), recons

# ═══════════════════════════════════════════════════════════════════════════
# RESULT → DATAFRAME
# ═══════════════════════════════════════════════════════════════════════════

def to_df(res):
    rows=[]
    for mn,imgs in res.items():
        for in_,ks in imgs.items():
            for k,mv in ks.items():
                if 'error' not in mv:
                    rows.append({'Method':mn,'Image':in_,'k':k,**mv})
    return pd.DataFrame(rows)

def summary(df, cols=None):
    if cols is None: cols=['SSIM','PSNR','NMSE','CR','QR','ETA','BPP','NNZC','TIME']
    cols=[c for c in cols if c in df.columns]
    return df.groupby('Method')[cols].agg(['mean','std']).round(4)

def pivot_k(df, metric='SSIM'):
    return df.pivot_table(values=metric,index='Method',columns='k',aggfunc='mean').round(4)

def pivot_img(df, metric='SSIM'):
    return df.pivot_table(values=metric,index='Method',columns='Image',aggfunc='mean').round(4)

def sigt(res, metric, methods, alpha=0.05):
    n=len(methods)
    S=np.zeros((n,n))
    for i,m1 in enumerate(methods):
        for j,m2 in enumerate(methods):
            if i==j: continue
            pairs=[(res[m1][img][k].get(metric),res[m2][img][k].get(metric))
                   for img in res.get(m1,{}) for k in res[m1][img]
                   if img in res.get(m2,{}) and k in res[m2][img]]
            pairs=[(a,b) for a,b in pairs if a is not None and b is not None]
            if len(pairs)<2: continue
            _,pv=ttest_rel([b for _,b in pairs],[a for a,_ in pairs],alternative='greater')
            S[i,j]=1 if pv<alpha else 0
    PreT=S.sum(1)/max(n-1,1)
    return S, PreT

# ═══════════════════════════════════════════════════════════════════════════
# PLOT HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def plot_lines(ax, df, x_col, y_col, title, xlabel, ylabel):
    style_ax(ax)
    methods = df['Method'].unique()
    for ci,mn in enumerate(methods):
        sub = df[df['Method']==mn].groupby(x_col)[y_col].mean()
        ax.plot(sub.index, sub.values, 'o-', color=ACCENT[ci%len(ACCENT)],
                label=mn, lw=1.8, ms=5, alpha=0.9)
    ax.set_title(title, fontsize=9, pad=6)
    ax.set_xlabel(xlabel, fontsize=8); ax.set_ylabel(ylabel, fontsize=8)
    ax.legend(fontsize=6, facecolor=PANEL, labelcolor=TEXT,
              edgecolor=GRID, framealpha=0.8)

def plot_heatmap(ax, data, title, fmt='.3f', fontsize=7):
    ax.set_facecolor(PANEL)
    sns.heatmap(data, ax=ax, cmap='YlOrRd', annot=True, fmt=fmt,
                linewidths=0.4, linecolor=BG,
                annot_kws={'size':fontsize,'color':'black'},
                cbar_kws={'shrink':0.8})
    ax.set_title(title, color=TEXT, fontsize=9, pad=6)
    ax.tick_params(colors=SUBTEXT, labelsize=7)
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right')
    plt.setp(ax.get_yticklabels(), rotation=0)

def plot_sigt(ax, S, methods, title, PreT=None):
    ax.set_facecolor(PANEL)
    labels=[m.replace('SpectralSIBS','Spec').replace('(','').replace(')','')
              .replace('p_ratio=','pr=') for m in methods]
    sub_title = title
    if PreT is not None:
        sub_title += '\nPreT: '+' | '.join(f'{l}={v:.2f}' for l,v in zip(labels,PreT))
    sns.heatmap(S, ax=ax, annot=True, fmt='.0f',
                xticklabels=labels, yticklabels=labels,
                cmap='Blues', cbar=False, linewidths=0.8, linecolor=BG)
    ax.set_title(sub_title, color=TEXT, fontsize=8, pad=6)
    ax.tick_params(colors=SUBTEXT, labelsize=7)
    plt.setp(ax.get_xticklabels(), rotation=40, ha='right')
    plt.setp(ax.get_yticklabels(), rotation=0)

# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    np.random.seed(42)
    IMG_SIZE = 64
    DELTA    = 4
    KS       = [2, 4, 6, 8, 10, 12, 14, 16]
    P_RATIO_LIST = [0.3, 0.5, 0.8]

    banner("SpectralSIBS — Comprehensive Comparison  [22 images · 5 experiments]")

    # ── Load images ────────────────────────────────────────────────────────
    section("Dataset Loading")
    dataset = DatasetLoader.load(IMG_SIZE)
    imgs    = list(dataset.keys())
    h       = IMG_SIZE

    # ── Build DCT dictionary ───────────────────────────────────────────────
    Phi = DictLearn.dct(h, 64)

    # ══════════════════════════════════════════════════════════════════════
    # EXP 1 — DCT dictionary: OMP vs SIBS1 vs SpectralSIBS variants
    # ══════════════════════════════════════════════════════════════════════
    banner("Experiment 1 — DCT Dictionary: OMP / SIBS1 / SpectralSIBS")
    facs1 = {
        'OMP':                    F_omp(Phi),
        'SIBS1':                  F_sibs1(Phi, DELTA),
        **{f'Spectral(pr={pr})': F_spec(Phi, DELTA, pr, 20)
           for pr in P_RATIO_LIST}
    }
    res1, recons1 = sweep(facs1, dataset, KS, 'Exp1 — DCT')
    df1 = to_df(res1)

    # ── Tables ────────────────────────────────────────────────────────────
    section("Exp 1 Tables")
    print_table(summary(df1), "Overall Summary (mean ± std)")
    print_table(pivot_k(df1,'SSIM'), "SSIM vs Sparsity k")
    print_table(pivot_k(df1,'PSNR'), "PSNR vs Sparsity k")
    print_table(pivot_k(df1,'NMSE'), "NMSE vs Sparsity k")
    print_table(pivot_k(df1,'BPP'),  "BPP vs Sparsity k")
    print_table(pivot_img(df1,'SSIM').T, "Per-Image Mean SSIM", max_rows=22)

    # ── Plots ─────────────────────────────────────────────────────────────
    section("Exp 1 Figures")
    methods1 = list(facs1.keys())

    # Fig 1a — SSIM/PSNR/NMSE/BPP vs k
    fig, axes = new_fig(2, 2, figsize=(14, 9))
    axes = axes.flatten()
    for ax,(yc,yl) in zip(axes,[('SSIM','SSIM'),('PSNR','PSNR (dB)'),
                                  ('NMSE','NMSE'),('BPP','BPP')]):
        plot_lines(ax, df1, 'k', yc, f'Exp1 — {yl} vs k', 'Sparsity k', yl)
    fig.suptitle('Experiment 1 — DCT Dictionary Metrics vs Sparsity',
                 color=TEXT, fontsize=13, y=1.01)
    plt.tight_layout(h_pad=2.5)
    save(fig, 'fig01_exp1_metrics_vs_k')

    # Fig 1b — per-image SSIM heatmap
    pimg1 = pivot_img(df1,'SSIM')
    fig, ax = new_fig(1,1, figsize=(max(16, len(pimg1.columns)*0.85), 4.5))
    plot_heatmap(ax, pimg1, 'Exp1 — Per-Image Mean SSIM (all k)')
    plt.tight_layout()
    save(fig, 'fig02_exp1_per_image_ssim')

    # Fig 1c — SigT for SSIM, PSNR, QR
    fig, axes = new_fig(1,3, figsize=(18,5))
    for ax, met in zip(axes, ['SSIM','PSNR','QR']):
        S, PreT = sigt(res1, met, methods1)
        plot_sigt(ax, S, methods1, f'SigT — {met}', PreT)
    fig.suptitle('Experiment 1 — Statistical Significance (SigT)',
                 color=TEXT, fontsize=13)
    plt.tight_layout()
    save(fig, 'fig03_exp1_sigt')

    # Fig 1d — box-plot of SSIM per method
    fig, ax = new_fig(1,1, figsize=(10,5))
    style_ax(ax)
    df1_box = df1[['Method','SSIM']].copy()
    df1_box['Method'] = df1_box['Method'].str.replace('SpectralSIBS','Spec')
    order = df1_box.groupby('Method')['SSIM'].median().sort_values(ascending=False).index
    sns.boxplot(data=df1_box, x='Method', y='SSIM', order=order,
                palette=ACCENT[:len(methods1)], ax=ax,
                medianprops={'color':'white','lw':2},
                whiskerprops={'color':SUBTEXT}, capprops={'color':SUBTEXT},
                flierprops={'marker':'o','markerfacecolor':SUBTEXT,'markersize':3})
    ax.set_title('Exp1 — SSIM Distribution per Method', fontsize=11)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    save(fig, 'fig04_exp1_ssim_boxplot')

    # ══════════════════════════════════════════════════════════════════════
    # EXP 2 — Image-learned atoms: KSVD / MOD / ODL / Spectral augmentation
    # ══════════════════════════════════════════════════════════════════════
    banner("Experiment 2 — Image-Learned Atom Augmentation")
    P_AUG = 8
    res2 = defaultdict(lambda: defaultdict(dict))
    recons2 = defaultdict(dict)

    bar2 = tqdm(dataset.items(), desc='  Exp2 — per image', ncols=80, colour='yellow',
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')
    for in_, img in bar2:
        bar2.set_postfix(img=in_)
        if img.ndim==3: img=color.rgb2gray(img)
        img = img.astype(np.float32)
        gamma = list(range(0, img.shape[1], DELTA))
        Psi   = img[:, gamma]
        sp    = min(3, P_AUG)

        d_ksvd = DictLearn.ksvd(Psi, P_AUG, n_iter=5, sp=sp)
        d_mod  = DictLearn.mod(Psi,  P_AUG, n_iter=5, sp=sp)
        d_odl  = DictLearn.odl(Psi,  P_AUG, n_iter=5, sp=sp)
        spec   = SpectralSIBSMethod(Phi, DELTA, 10, 1.0, P_AUG)
        spec._pca(img)

        def enc_aux(aux, k):
            Pa  = np.hstack([Phi, aux])
            Pan = Pa/(np.linalg.norm(Pa,axis=0,keepdims=True)+1e-10)
            oms=[]; R=np.zeros_like(img)
            for i in range(img.shape[1]):
                c,idx = OMP.static(img[:,i], Pa, k)
                if len(idx): R[:,i]=Pa[:,idx]@c
                oms.append(c)
            return oms, np.clip(R,0,1)

        for k in KS:
            bar2.set_postfix(img=in_, k=k)
            for tag,aux in [('KSVD_aug',d_ksvd),('MOD_aug',d_mod),
                            ('ODL_aug',d_odl),('Spectral_aug',spec.PCs)]:
                if aux is None or aux.shape[1]==0: continue
                t0=time()
                oms,rec = enc_aux(aux, k)
                elapsed=time()-t0
                dummy = Base(np.hstack([Phi,aux]), k)
                mv = dummy.metrics(img, rec, oms, extra=aux.shape[1])
                mv['TIME']=round(elapsed,4)
                res2[tag][in_][k]=mv
                if k==KS[-1]: recons2[tag][in_]=rec
    bar2.close()

    df2 = to_df(res2)

    section("Exp 2 Tables")
    print_table(summary(df2), "Exp2 — Overall Summary")
    print_table(pivot_k(df2,'SSIM'), "Exp2 — SSIM vs k")
    print_table(pivot_k(df2,'PSNR'), "Exp2 — PSNR vs k")
    print_table(pivot_img(df2,'SSIM').T, "Exp2 — Per-Image SSIM", max_rows=22)

    section("Exp 2 Figures")
    fig, axes = new_fig(2,2, figsize=(14,9))
    axes=axes.flatten()
    for ax,(yc,yl) in zip(axes,[('SSIM','SSIM'),('PSNR','PSNR (dB)'),
                                  ('NMSE','NMSE'),('BPP','BPP')]):
        plot_lines(ax, df2, 'k', yc, f'Exp2 — {yl} vs k', 'Sparsity k', yl)
    fig.suptitle('Experiment 2 — Image-Learned Atom Augmentation', color=TEXT, fontsize=13)
    plt.tight_layout(h_pad=2.5)
    save(fig, 'fig05_exp2_metrics_vs_k')

    pimg2 = pivot_img(df2,'SSIM')
    fig, ax = new_fig(1,1, figsize=(max(16,len(pimg2.columns)*0.85), 4))
    plot_heatmap(ax, pimg2, 'Exp2 — Per-Image Mean SSIM')
    plt.tight_layout()
    save(fig, 'fig06_exp2_per_image_ssim')

    # ══════════════════════════════════════════════════════════════════════
    # EXP 3 — Pre-learned dictionaries
    # ══════════════════════════════════════════════════════════════════════
    banner("Experiment 3 — Pre-Learned Dictionaries (KSVD·MOD·ODL·BKSVD·BSSDL·RBDL)")
    train_imgs = list(dataset.values())[:12]
    test_ds    = {k:v for k,v in list(dataset.items())[12:]}

    cols_train=[]
    for im in train_imgs:
        if im.ndim==3: im=color.rgb2gray(im)
        im=im.astype(np.float32)
        cols_train.append(im-im.mean(0,keepdims=True))
    train_data = np.hstack(cols_train)

    DICT_SPECS = [('KSVD',  DictLearn.ksvd, 5),
                  ('MOD',   DictLearn.mod,   5),
                  ('ODL',   DictLearn.odl,   5),
                  ('BKSVD', DictLearn.mod,   3),
                  ('BSSDL', DictLearn.mod,   4),
                  ('RBDL',  DictLearn.mod,   5)]

    learned = {}
    bar3a = tqdm(DICT_SPECS, desc='  Exp3 — training dicts', ncols=72, colour='magenta')
    for dn, fn, sp in bar3a:
        bar3a.set_postfix(dict=dn)
        d = fn(train_data, 64, n_iter=8, sp=sp)
        learned[dn] = DictLearn._norm(d)
    bar3a.close()

    res3=defaultdict(lambda:defaultdict(dict)); recons3=defaultdict(dict)
    bar3b = tqdm(learned.items(), desc='  Exp3 — encoding', ncols=72, colour='magenta')
    for dn, D in bar3b:
        bar3b.set_postfix(dict=dn)
        facs={f'OMP_{dn}':     F_omp(D),
              f'SIBS1_{dn}':   F_sibs1(D,DELTA),
              f'Spectral_{dn}':F_spec(D,DELTA,0.5,16)}
        r, rc = sweep(facs, test_ds, KS, f'  {dn}')
        for mn,inner in r.items():
            for in_,kd in inner.items():
                for k,mv in kd.items(): res3[mn][in_][k]=mv
        for mn,inner in rc.items():
            for in_,im in inner.items(): recons3[mn][in_]=im
    bar3b.close()

    df3=to_df(res3)

    section("Exp 3 Tables")
    print_table(summary(df3), "Exp3 — Overall Summary")
    # Grouped by base method and dict
    df3['Base']  = df3['Method'].str.extract(r'^(OMP|SIBS1|Spectral)')
    df3['Dict']  = df3['Method'].str.extract(r'_(KSVD|MOD|ODL|BKSVD|BSSDL|RBDL)')
    grp3 = df3.groupby(['Dict','Base'])['SSIM'].mean().unstack('Base').round(4)
    print_table(grp3, "Exp3 — Mean SSIM grouped by Dict × Base-Method")
    grp3p = df3.groupby(['Dict','Base'])['PSNR'].mean().unstack('Base').round(4)
    print_table(grp3p, "Exp3 — Mean PSNR grouped by Dict × Base-Method")

    section("Exp 3 Figures")
    # Grouped bar — SSIM
    fig, axes = new_fig(1,2, figsize=(14,5))
    for ax,(met,yl) in zip(axes,[('SSIM','SSIM'),('PSNR','PSNR (dB)')]):
        style_ax(ax)
        dnames = list(learned.keys())
        x = np.arange(len(dnames)); W=0.25
        for bi,(base,col) in enumerate([('OMP',ACCENT[0]),('SIBS1',ACCENT[1]),('Spectral',ACCENT[2])]):
            vals=[df3[(df3['Base']==base)&(df3['Dict']==dn)][met].mean()
                  if len(df3[(df3['Base']==base)&(df3['Dict']==dn)])>0 else 0
                  for dn in dnames]
            ax.bar(x+bi*W, vals, W, label=base, color=col, alpha=0.85, edgecolor=BG, lw=0.5)
        ax.set_xticks(x+W); ax.set_xticklabels(dnames, color=SUBTEXT, fontsize=8)
        ax.set_title(f'Exp3 — Mean {yl} by Dictionary', fontsize=10)
        ax.set_ylabel(yl, fontsize=8)
        ax.legend(facecolor=PANEL, labelcolor=TEXT, fontsize=8, edgecolor=GRID)
    fig.suptitle('Experiment 3 — Pre-Learned Dictionaries', color=TEXT, fontsize=13)
    plt.tight_layout()
    save(fig, 'fig07_exp3_dict_bars')

    pimg3 = pivot_img(df3[df3['Base']=='Spectral'],'SSIM') if 'SSIM' in df3.columns else None
    if pimg3 is not None and len(pimg3)>0:
        fig, ax = new_fig(1,1, figsize=(max(12,len(pimg3.columns)*0.9), 4))
        plot_heatmap(ax, pimg3, 'Exp3 — Spectral Method: Per-Image SSIM across Dicts')
        plt.tight_layout()
        save(fig, 'fig08_exp3_spectral_per_image')

    # ══════════════════════════════════════════════════════════════════════
    # EXP 4 — Rate-distortion with overhead
    # ══════════════════════════════════════════════════════════════════════
    banner("Experiment 4 — Rate-Distortion with Overhead")
    B_A=8
    facs4 = {
        'OMP':           F_omp(Phi),
        'SIBS1':         F_sibs1(Phi,DELTA),
        'Spectral_p8':   F_spec(Phi,DELTA,1.0,8),
        'Spectral_p16':  F_spec(Phi,DELTA,1.0,16),
    }
    res4, recons4 = sweep(facs4, dataset, KS, 'Exp4 — RD')
    df4 = to_df(res4)

    # Add overhead columns
    def add_overhead(df, res, B_A=8):
        rows=[]
        for mn,imgs_ in res.items():
            for in_,ks_ in imgs_.items():
                for k,mv in ks_.items():
                    if 'error' in mv: continue
                    extra=0
                    if 'p8' in mn: extra=8
                    elif 'p16' in mn: extra=16
                    bpp_oh = (extra*B_A)/IMG_SIZE
                    rows.append({'Method':mn,'Image':in_,'k':k,
                                 **mv,'BPP_OH':bpp_oh,'BPP_TOT':mv['BPP']+bpp_oh})
        return pd.DataFrame(rows)
    df4x = add_overhead(df4, res4)

    section("Exp 4 Tables")
    print_table(summary(df4x,['SSIM','PSNR','BPP','BPP_OH','BPP_TOT']),
                "Exp4 — RD Summary (with overhead)")
    print_table(pivot_k(df4x,'SSIM'), "Exp4 — SSIM vs k")
    print_table(pivot_k(df4x,'BPP_TOT'), "Exp4 — Total BPP vs k")

    section("Exp 4 Figures")
    fig, axes = new_fig(1,3, figsize=(18,5))
    # SSIM vs BPP (base)
    plot_lines(axes[0], df4x.groupby(['Method','BPP'])['SSIM'].mean().reset_index(),
               'BPP','SSIM','Exp4 — SSIM vs BPP (base)','BPP','SSIM')
    # SSIM vs total BPP
    plot_lines(axes[1], df4x.groupby(['Method','BPP_TOT'])['SSIM'].mean().reset_index(),
               'BPP_TOT','SSIM','Exp4 — SSIM vs Total BPP (w/ overhead)','Total BPP','SSIM')
    # PSNR vs k
    plot_lines(axes[2], df4x, 'k','PSNR','Exp4 — PSNR vs k','Sparsity k','PSNR (dB)')
    fig.suptitle('Experiment 4 — Rate-Distortion', color=TEXT, fontsize=13)
    plt.tight_layout()
    save(fig, 'fig09_exp4_rd_curves')

    # ══════════════════════════════════════════════════════════════════════
    # EXP 5 — Parameter sensitivity
    # ══════════════════════════════════════════════════════════════════════
    banner("Experiment 5 — Parameter Sensitivity")
    sens_img = list(dataset.values())[0].astype(np.float32)

    res5={'vary_p':{},'vary_delta':{},'vary_p_ratio':{}}
    bar5 = tqdm(range(1,21), desc='  Exp5 — vary p', ncols=72, colour='cyan')
    for p in bar5:
        bar5.set_postfix(p=p)
        m=SpectralSIBSMethod(Phi,DELTA,10,1.0,p)
        oms,_,rec=m.encode(sens_img)
        mv=m.metrics(sens_img,rec,oms,extra=p)
        mv['EXP_VAR']=float(np.sum(m.var_ratio))
        mv['N_PCS']=m.PCs.shape[1]
        res5['vary_p'][p]=mv
    bar5.close()

    bar5b = tqdm([1,2,4,8,16,32], desc='  Exp5 — vary delta', ncols=72, colour='cyan')
    for delta in bar5b:
        bar5b.set_postfix(delta=delta)
        m=SpectralSIBSMethod(Phi,delta,10,0.5,20)
        oms,_,rec=m.encode(sens_img)
        mv=m.metrics(sens_img,rec,oms,extra=m.PCs.shape[1])
        mv['N_PCS']=m.PCs.shape[1]
        res5['vary_delta'][delta]=mv
    bar5b.close()

    pr_vals=np.round(np.arange(0.1,1.01,0.1),1)
    bar5c = tqdm(pr_vals, desc='  Exp5 — vary p_ratio', ncols=72, colour='cyan')
    for pr in bar5c:
        bar5c.set_postfix(p_ratio=pr)
        m=SpectralSIBSMethod(Phi,DELTA,10,float(pr),20)
        oms,_,rec=m.encode(sens_img)
        mv=m.metrics(sens_img,rec,oms,extra=m.PCs.shape[1])
        mv['N_PCS']=m.PCs.shape[1]
        res5['vary_p_ratio'][float(pr)]=mv
    bar5c.close()

    df5p = pd.DataFrame(res5['vary_p']).T.rename_axis('p')
    df5d = pd.DataFrame(res5['vary_delta']).T.rename_axis('delta')
    df5r = pd.DataFrame(res5['vary_p_ratio']).T.rename_axis('p_ratio')

    section("Exp 5 Tables")
    print_table(df5p[['SSIM','PSNR','NMSE','BPP','EXP_VAR','N_PCS']].round(4),
                "Exp5 — Vary p (number of PCs)")
    print_table(df5d[['SSIM','PSNR','NMSE','BPP','N_PCS']].round(4),
                "Exp5 — Vary delta (sampling stride)")
    print_table(df5r[['SSIM','PSNR','NMSE','BPP','N_PCS']].round(4),
                "Exp5 — Vary p_ratio")

    section("Exp 5 Figures")
    fig, axes = new_fig(2,3, figsize=(18,9))
    axes=axes.flatten()

    metrics5=[('SSIM','SSIM'),('PSNR','PSNR (dB)'),('EXP_VAR','Explained Variance')]
    for ax,(col,lab) in zip(axes[:3], metrics5):
        style_ax(ax)
        y=df5p[col].astype(float) if col in df5p.columns else pd.Series()
        if len(y): ax.plot(df5p.index, y.values, 'o-', color=ACCENT[0], lw=2, ms=6)
        ax.set_title(f'Vary p — {lab}', fontsize=9); ax.set_xlabel('p'); ax.set_ylabel(lab)
    for ax,(src,xlab,col,lab) in zip(axes[3:], [
            (df5d,'delta','SSIM','SSIM'),
            (df5d,'delta','PSNR','PSNR (dB)'),
            (df5r,'p_ratio','SSIM','SSIM')]):
        style_ax(ax)
        y=src[col].astype(float) if col in src.columns else pd.Series()
        x=src.index.astype(float)
        if len(y): ax.plot(x, y.values, 's-', color=ACCENT[1], lw=2, ms=6)
        ax.set_title(f'Vary {xlab} — {lab}', fontsize=9)
        ax.set_xlabel(xlab); ax.set_ylabel(lab)
    fig.suptitle('Experiment 5 — Parameter Sensitivity', color=TEXT, fontsize=13)
    plt.tight_layout(h_pad=3)
    save(fig, 'fig10_exp5_sensitivity')

    # ══════════════════════════════════════════════════════════════════════
    # VISUAL RECONSTRUCTION PANELS
    # ══════════════════════════════════════════════════════════════════════
    banner("Visual Reconstruction Comparison")
    vis_names = [n for n in ['camera','chelsea','moon','coins','rocket',
                              'checkerboard','coffee','hubble'] if n in dataset][:6]
    vis_methods_ordered = ['OMP','SIBS1',
                           f'Spectral(pr={P_RATIO_LIST[0]})',
                           f'Spectral(pr={P_RATIO_LIST[1]})',
                           f'Spectral(pr={P_RATIO_LIST[2]})']
    cols_vis = ['Original'] + vis_methods_ordered

    fig = plt.figure(figsize=(len(cols_vis)*2.6, len(vis_names)*2.6), facecolor=BG)
    gs  = gridspec.GridSpec(len(vis_names), len(cols_vis), figure=fig,
                            hspace=0.06, wspace=0.04)

    bar_vis = tqdm(enumerate(vis_names), total=len(vis_names),
                   desc='  Building visual panel', ncols=72, colour='white')
    for ri, in_ in bar_vis:
        bar_vis.set_postfix(img=in_)
        orig = dataset[in_]
        for ci, tag in enumerate(cols_vis):
            ax = fig.add_subplot(gs[ri, ci])
            ax.set_facecolor(BG)
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_color(GRID)

            if tag == 'Original':
                ax.imshow(orig, cmap='gray', vmin=0, vmax=1)
                if ri == 0: ax.set_title('Original', color=TEXT, fontsize=8, pad=4)
                ax.set_ylabel(in_, color=SUBTEXT, fontsize=8, labelpad=3)
            else:
                im_data = recons1.get(tag,{}).get(in_)
                if im_data is not None:
                    ax.imshow(im_data, cmap='gray', vmin=0, vmax=1)
                    try:
                        sv=ssim(orig.astype(np.float32), im_data, data_range=1.)
                        pv_=-10*math.log10(float(np.mean((orig-im_data)**2))+1e-12)
                        ax.set_xlabel(f'SSIM={sv:.3f}\nPSNR={pv_:.1f}',
                                      color=SUBTEXT, fontsize=6, labelpad=2)
                    except: pass
                    if ri==0:
                        label=(tag.replace('SpectralSIBS','Spec')
                                  .replace('Spectral','Spec')
                                  .replace('(','').replace(')',''))
                        ax.set_title(label, color=TEXT, fontsize=7, pad=4)
                else:
                    ax.text(0.5,0.5,'N/A', ha='center', va='center',
                            color=SUBTEXT, transform=ax.transAxes, fontsize=10)
    bar_vis.close()
    fig.suptitle(f'Visual Reconstruction Comparison  (k={KS[-1]})',
                 color=TEXT, fontsize=13, y=1.005)
    save(fig, 'fig11_visual_recons')

    # ── Full dataset gallery ───────────────────────────────────────────────
    section("Dataset Gallery")
    ncols=6; nrows=math.ceil(len(imgs)/ncols)
    fig,axes=new_fig(nrows,ncols,figsize=(ncols*2.3,nrows*2.4))
    axes=axes.flatten()
    for ai,in_ in enumerate(imgs):
        ax=axes[ai]
        ax.imshow(dataset[in_],cmap='gray',vmin=0,vmax=1)
        ax.set_title(in_,color=SUBTEXT,fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_facecolor(BG)
    for ai in range(len(imgs),len(axes)): axes[ai].axis('off')
    fig.suptitle(f'Full Dataset — {len(imgs)} Images  (size {IMG_SIZE}×{IMG_SIZE})',
                 color=TEXT, fontsize=13)
    plt.tight_layout()
    save(fig, 'fig00_dataset_gallery')

    # ══════════════════════════════════════════════════════════════════════
    # MASTER COMPARISON TABLE (all methods, all metrics, avg over all k & images)
    # ══════════════════════════════════════════════════════════════════════
    banner("Master Comparison Table (all experiments)")
    master_rows=[]
    for df_, exp in [(df1,'Exp1-DCT'),(df2,'Exp2-AugLearn'),(df4,'Exp4-RD')]:
        if len(df_)==0: continue
        g=df_.groupby('Method')[['SSIM','PSNR','NMSE','CR','QR','ETA','BPP','NNZC','TIME']].mean()
        g.insert(0,'Experiment',exp)
        master_rows.append(g.reset_index())
    master_df=pd.concat(master_rows,ignore_index=True) if master_rows else pd.DataFrame()
    if len(master_df):
        print_table(master_df.sort_values(['Experiment','SSIM'],ascending=[True,False]),
                    "Master Comparison — Mean over all images & k")

    # ══════════════════════════════════════════════════════════════════════
    # SAVE EXCEL + CSV
    # ══════════════════════════════════════════════════════════════════════
    section("Saving Persistent Tables")
    excel_path = f"{OUT}/spectral_sibs_all_results.xlsx"
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for df_,_raw,label in [(df1,'Exp1_DCT_raw','Exp1_DCT'),
                               (df2,'Exp2_Learned_raw','Exp2_Learned'),
                               (df3,'Exp3_PreLearned_raw','Exp3_PreLearned'),
                               (df4,'Exp4_RD_raw','Exp4_RD'),
                               (df4x,'Exp4_RD_overhead','Exp4_RD_ovhd')]:
            if len(df_)==0: continue
            df_.to_excel(writer,sheet_name=label[:31],index=False)
            summary(df_).to_excel(writer,sheet_name=f'{label[:22]}_Summ')
            pivot_k(df_,'SSIM').to_excel(writer,sheet_name=f'{label[:22]}_SSIM_k')
            pivot_k(df_,'PSNR').to_excel(writer,sheet_name=f'{label[:22]}_PSNR_k')
            pivot_img(df_,'SSIM').to_excel(writer,sheet_name=f'{label[:22]}_SSIM_img')

        df5p.to_excel(writer,sheet_name='Exp5_vary_p')
        df5d.to_excel(writer,sheet_name='Exp5_vary_delta')
        df5r.to_excel(writer,sheet_name='Exp5_vary_pratio')
        if len(master_df): master_df.to_excel(writer,sheet_name='Master_Comparison',index=False)

        for met in ['SSIM','PSNR','QR']:
            S,PreT=sigt(res1,met,methods1)
            df_sig=pd.DataFrame(S,index=methods1,columns=methods1)
            df_sig.loc['PreT']=PreT
            df_sig.to_excel(writer,sheet_name=f'SigT_Exp1_{met}')

    print(f"  ✓ Excel: spectral_sibs_all_results.xlsx")
    for tag,df_ in [('exp1_dct',df1),('exp2_learned',df2),
                    ('exp3_prelearned',df3),('exp4_rd',df4x)]:
        if len(df_)==0: continue
        df_.to_csv(f"{OUT}/{tag}_full.csv",index=False)
        summary(df_).to_csv(f"{OUT}/{tag}_summary.csv")
        print(f"  ✓ CSV: {tag}_full.csv + {tag}_summary.csv")

    # ══════════════════════════════════════════════════════════════════════
    # FINAL FILE LIST
    # ══════════════════════════════════════════════════════════════════════
    banner("All Output Files")
    files=sorted(os.listdir(OUT))
    rows_f=[[f, f"{os.path.getsize(os.path.join(OUT,f))//1024} KB"] for f in files]
    print(tabulate(rows_f,headers=['Filename','Size'],tablefmt='rounded_outline'))

    print("\n  ✅  All experiments complete.\n")

if __name__ == '__main__':
    main()


╔══════════════════════════════════════════════════════════════════════╗
║  SpectralSIBS — Comprehensive Comparison  [22 images · 5 experiments]║
╚══════════════════════════════════════════════════════════════════════╝

────────────────────────────────────────────────────────────────────────
  Dataset Loading
────────────────────────────────────────────────────────────────────────


Loading images: 100%|███| 22/22 [00:00<00:00, 31.42it/s, img=microangio]



  ✓ Loaded 22 images: camera, astronaut, brick, rocket, chelsea, coins, horse, moon, clock, coffee, grass, gravel, text_img, page, retina, hubble, checkerboard, colorwheel, cat, blobs, phantom, microangio

╔══════════════════════════════════════════════════════════════════════╗
║  Experiment 1 — DCT Dictionary: OMP / SIBS1 / SpectralSIBS           ║
╚══════════════════════════════════════════════════════════════════════╝


  Exp1 — DCT: 100%|███████████████████████████| 880/880 [00:52<00:00, 16.69it/s]



────────────────────────────────────────────────────────────────────────
  Exp 1 Tables
────────────────────────────────────────────────────────────────────────

  ▶ Overall Summary (mean ± std)
╭──────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬──────────────────┬─────────────────┬──────────────────┬─────────────────┬───────────────────┬──────────────────┬───────────────────┬──────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────╮
│ Method           │   ('SSIM', 'mean') │   ('SSIM', 'std') │   ('PSNR', 'mean') │   ('PSNR', 'std') │   ('NMSE', 'mean') │   ('NMSE', 'std') │   ('CR', 'mean') │   ('CR', 'std') │   ('QR', 'mean') │   ('QR', 'std') │   ('ETA', 'mean') │   ('ETA', 'std') │   ('BPP', 'mean') │   ('BPP', 'std') │   ('NNZC', 'mean') │   ('NNZC', 'std') │   ('TIME', 'mean') │   ('TIME', 'std') │
├──────────────────┼────────────────────┼───────

...


────────────────────────────────────────────────────────────────────────
  Exp 4 Tables
────────────────────────────────────────────────────────────────────────

  ▶ Exp4 — RD Summary (with overhead)
╭──────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┬──────────────────┬──────────────────────┬─────────────────────┬───────────────────────┬──────────────────────╮
│ Method       │   ('SSIM', 'mean') │   ('SSIM', 'std') │   ('PSNR', 'mean') │   ('PSNR', 'std') │   ('BPP', 'mean') │   ('BPP', 'std') │   ('BPP_OH', 'mean') │   ('BPP_OH', 'std') │   ('BPP_TOT', 'mean') │   ('BPP_TOT', 'std') │
├──────────────┼────────────────────┼───────────────────┼────────────────────┼───────────────────┼───────────────────┼──────────────────┼──────────────────────┼─────────────────────┼───────────────────────┼──────────────────────┤
│ OMP          │             0.6382 │            0.1549 │            22.9331 │            6.6413 │           

  Exp5 — vary p_ratio: 100%|█| 10/10 [00:00<00:00, 15.98it/s, p_ratio=1]



────────────────────────────────────────────────────────────────────────
  Exp 5 Tables
────────────────────────────────────────────────────────────────────────

  ▶ Exp5 — Vary p (number of PCs)
╭─────┬────────┬─────────┬────────┬────────┬───────────┬─────────╮
│   p │   SSIM │    PSNR │   NMSE │    BPP │   EXP_VAR │   N_PCS │
├─────┼────────┼─────────┼────────┼────────┼───────────┼─────────┤
│   1 │ 0.5668 │ 21.8346 │ 0.0197 │ 2.5000 │    0.5856 │  1.0000 │
│   2 │ 0.5961 │ 22.9119 │ 0.0154 │ 2.5000 │    0.8018 │  2.0000 │
│   3 │ 0.6708 │ 24.8189 │ 0.0099 │ 2.5000 │    0.8957 │  3.0000 │
│   4 │ 0.7138 │ 26.3085 │ 0.0070 │ 2.5000 │    0.9344 │  4.0000 │
│   5 │ 0.7334 │ 26.9342 │ 0.0061 │ 2.5000 │    0.9546 │  5.0000 │
│   6 │ 0.7551 │ 27.2626 │ 0.0057 │ 2.5000 │    0.9665 │  6.0000 │
│   7 │ 0.7686 │ 27.5761 │ 0.0053 │ 2.5000 │    0.9772 │  7.0000 │
│   8 │ 0.7815 │ 27.8512 │ 0.0049 │ 2.5000 │    0.9841 │  8.0000 │
│   9 │ 0.7943 │ 28.0537 │ 0.0047 │ 2.5000 │    0.9894 │  9.0000 │

  Building visual panel: 100%|█| 6/6 [00:00<00:00,  7.19it/s, img=checke


  ✓ saved fig11_visual_recons.png

────────────────────────────────────────────────────────────────────────
  Dataset Gallery
────────────────────────────────────────────────────────────────────────
  ✓ saved fig00_dataset_gallery.png

╔══════════════════════════════════════════════════════════════════════╗
║  Master Comparison Table (all experiments)                           ║
╚══════════════════════════════════════════════════════════════════════╝

  ▶ Master Comparison — Mean over all images & k
╭────┬──────────────────┬───────────────┬────────┬─────────┬────────┬────────┬────────┬─────────┬────────┬────────┬────────╮
│    │ Method           │ Experiment    │   SSIM │    PSNR │   NMSE │     CR │     QR │     ETA │    BPP │   NNZC │   TIME │
├────┼──────────────────┼───────────────┼────────┼─────────┼────────┼────────┼────────┼─────────┼────────┼────────┼────────┤
│  4 │ Spectral(pr=0.8) │ Exp1-DCT      │ 0.8632 │ 35.4392 │ 0.0151 │ 0.1659 │ 0.1372 │ 16.5904 │ 2.2500 │ 9.0000 │ 0.05

In [31]:
# CELL 1 - Imports, paths, dark theme
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
warnings.filterwarnings('ignore')

DATA_DIR = '/mnt/user-data/outputs'
EXCEL    = os.path.join(DATA_DIR, 'spectral_sibs_all_results.xlsx')

BG     = '#0b0f1a'
PANEL  = '#131929'
GRID   = '#1e2d45'
TEXT   = '#d0e8ff'
SUB    = '#7ba3c8'
ACCENT = ['#00d4ff','#ff6b35','#7fff6b','#ffcc00','#cc44ff',
           '#ff4488','#44ffee','#ff8800','#aaffcc','#ff2255']

plt.rcParams.update({
    'figure.facecolor': BG,   'axes.facecolor': PANEL, 'axes.edgecolor': GRID,
    'axes.labelcolor': TEXT,  'axes.grid': True,        'grid.color': GRID,
    'grid.alpha': 0.5,        'xtick.color': SUB,       'ytick.color': SUB,
    'text.color': TEXT,       'legend.facecolor': PANEL,'legend.edgecolor': GRID,
    'legend.labelcolor': TEXT,'font.size': 9,            'axes.titlesize': 10,
    'axes.titlecolor': TEXT,
})

assert os.path.exists(DATA_DIR), f'DATA_DIR not found: {DATA_DIR}'
assert os.path.exists(EXCEL),    f'Excel file not found: {EXCEL}'
print('Imports OK  |  Data dir:', DATA_DIR)

Imports OK  |  Data dir: /mnt/user-data/outputs


In [32]:
# CELL 2 - Load all data
xl = pd.ExcelFile(EXCEL)

df1 = pd.read_csv(os.path.join(DATA_DIR, 'exp1_dct_full.csv'))
df2 = pd.read_csv(os.path.join(DATA_DIR, 'exp2_learned_full.csv'))
df3 = pd.read_csv(os.path.join(DATA_DIR, 'exp3_prelearned_full.csv'))
df4 = pd.read_csv(os.path.join(DATA_DIR, 'exp4_rd_full.csv'))
df3['Base'] = df3['Method'].str.extract(r'^(OMP|SIBS1|Spectral)')
df3['Dict'] = df3['Method'].str.extract(r'_(KSVD|MOD|ODL|BKSVD|BSSDL|RBDL)')

df5p      = xl.parse('Exp5_vary_p').set_index('p')
df5d      = xl.parse('Exp5_vary_delta').set_index('delta')
df5r      = xl.parse('Exp5_vary_pratio').set_index('p_ratio')
master    = xl.parse('Master_Comparison')
sigt_ssim = xl.parse('SigT_Exp1_SSIM', index_col=0)
sigt_psnr = xl.parse('SigT_Exp1_PSNR', index_col=0)
sigt_qr   = xl.parse('SigT_Exp1_QR',   index_col=0)

print(f'Exp1  {len(df1):>5,} rows | {df1.Method.nunique()} methods | {df1.Image.nunique()} images')
print(f'Exp2  {len(df2):>5,} rows | {df2.Method.nunique()} methods')
print(f'Exp3  {len(df3):>5,} rows | {df3.Method.nunique()} methods')
print(f'Exp4  {len(df4):>5,} rows | {df4.Method.nunique()} methods | BPP_TOT present: {"BPP_TOT" in df4.columns}')
print(f'Exp5  vary_p={len(df5p)}  vary_delta={len(df5d)}  vary_pratio={len(df5r)}')
print(f'Master {len(master)} rows')

Exp1    880 rows | 5 methods | 22 images
Exp2    704 rows | 4 methods
Exp3  1,440 rows | 18 methods
Exp4    704 rows | 4 methods | BPP_TOT present: True
Exp5  vary_p=20  vary_delta=6  vary_pratio=10
Master 13 rows


In [33]:
# CELL 3 - Exp 1: Metrics vs sparsity k
methods1 = df1['Method'].unique()

fig, axes = plt.subplots(1, 4, figsize=(20, 5), facecolor=BG)
for ax, (col, lab) in zip(axes, [('SSIM','SSIM'), ('PSNR','PSNR (dB)'),
                                   ('NMSE','NMSE'),  ('BPP','BPP')]):
    for ci, mn in enumerate(methods1):
        sub = df1[df1['Method']==mn].groupby('k')[col].mean()
        lbl = mn.replace('Spectral','Spec').replace('(','').replace(')','').replace('SIBS1','SIBS')
        ax.plot(sub.index, sub.values, 'o-', color=ACCENT[ci%10], label=lbl, lw=2, ms=5)
    ax.set_xlabel('Sparsity k'); ax.set_ylabel(lab)
    ax.set_title(f'{lab} vs k'); ax.legend(fontsize=7)

fig.suptitle('Experiment 1 - DCT Dictionary: Metrics vs Sparsity k', color=TEXT, fontsize=13)
plt.tight_layout()
plt.show()

In [34]:
# CELL 4 - Exp 1: Per-image SSIM heatmap
pivot1 = df1.pivot_table(values='SSIM', index='Method', columns='Image', aggfunc='mean').round(3)

fig, ax = plt.subplots(figsize=(max(16, len(pivot1.columns)*0.85), 5), facecolor=BG)
ax.set_facecolor(PANEL)
sns.heatmap(pivot1, ax=ax, cmap='YlOrRd', annot=True, fmt='.3f',
            linewidths=0.4, linecolor=BG,
            annot_kws={'size': 6, 'color': 'black'},
            cbar_kws={'shrink': 0.8})
ax.set_title('Exp 1 - Per-Image Mean SSIM (averaged over all k)', color=TEXT, fontsize=12)
plt.xticks(rotation=40, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

top = pivot1.stack().reset_index()
top.columns = ['Method','Image','SSIM']
print('Top 10 method x image combos by SSIM:')
print(top.nlargest(10, 'SSIM').to_string(index=False))

Top 10 method x image combos by SSIM:
          Method        Image  SSIM
Spectral(pr=0.3) checkerboard 1.000
Spectral(pr=0.5) checkerboard 1.000
Spectral(pr=0.8) checkerboard 1.000
Spectral(pr=0.5)        clock 0.992
Spectral(pr=0.8)        clock 0.992
Spectral(pr=0.3)        clock 0.988
Spectral(pr=0.5)   colorwheel 0.984
Spectral(pr=0.8)   colorwheel 0.984
Spectral(pr=0.3)   colorwheel 0.971
Spectral(pr=0.8)       rocket 0.962


In [35]:
# CELL 5 - Exp 1: Box + Violin SSIM distributions
order    = df1.groupby('Method')['SSIM'].median().sort_values(ascending=False).index
labels_s = [m.replace('Spectral','Spec').replace('(','').replace(')','') for m in order]
data_box = [df1[df1['Method']==m]['SSIM'].values for m in order]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)

bp = axes[0].boxplot(data_box, patch_artist=True,
                     medianprops={'color':'white','lw':2},
                     whiskerprops={'color':SUB}, capprops={'color':SUB},
                     flierprops={'marker':'o','markerfacecolor':SUB,'markersize':3})
for patch, col in zip(bp['boxes'], ACCENT):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[0].set_xticks(range(1, len(order)+1))
axes[0].set_xticklabels(labels_s, rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('SSIM'); axes[0].set_title('SSIM Boxplot')

parts = axes[1].violinplot(data_box, showmedians=True)
for pc, col in zip(parts['bodies'], ACCENT):
    pc.set_facecolor(col); pc.set_alpha(0.65)
for key in ('cmedians','cbars','cmins','cmaxes'):
    parts[key].set_color('white' if key=='cmedians' else SUB)
    if key == 'cmedians': parts[key].set_lw(2)
axes[1].set_xticks(range(1, len(order)+1))
axes[1].set_xticklabels(labels_s, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('SSIM'); axes[1].set_title('SSIM Violin')

fig.suptitle('Experiment 1 - SSIM Distributions per Method', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

In [36]:
# CELL 6 - Exp 1: Statistical significance (SigT)
def draw_sigt(ax, ds, title):
    mat  = ds.drop(index='PreT', errors='ignore')
    mat  = mat[[c for c in mat.columns if c != 'PreT']].astype(float)
    lbls = [l.replace('Spectral','Spec').replace('(','').replace(')','') for l in mat.index]
    sns.heatmap(mat.values, ax=ax, cmap='Blues', annot=True, fmt='.0f',
                xticklabels=lbls, yticklabels=lbls, cbar=False,
                linewidths=0.6, linecolor=BG)
    ax.set_title(title, color=TEXT, fontsize=10); ax.tick_params(labelsize=7)
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
    plt.setp(ax.get_yticklabels(), rotation=0)
    if 'PreT' in ds.index:
        pret = ds.loc['PreT', [c for c in ds.columns if c != 'PreT']].astype(float)
        ax.set_xlabel('PreT: ' + ' | '.join(f'{l}={v:.2f}' for l, v in zip(lbls, pret)),
                      color=SUB, fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(19, 6), facecolor=BG)
for ax, ds, t in zip(axes,
                     [sigt_ssim, sigt_psnr, sigt_qr],
                     ['SigT - SSIM', 'SigT - PSNR', 'SigT - QR']):
    ax.set_facecolor(PANEL); draw_sigt(ax, ds, t)

fig.suptitle('Exp 1 - Statistical Significance  (1 = column method significantly beats row)',
             color=TEXT, fontsize=12)
plt.tight_layout(); plt.show()

In [37]:
# CELL 7 - Exp 1: Rate-distortion scatter
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)

for ci, mn in enumerate(methods1):
    sub = df1[df1['Method']==mn]; col = ACCENT[ci%10]
    lbl = mn.replace('Spectral','Spec').replace('(','').replace(')','').replace('SIBS1','SIBS')
    for ax, ycol in zip(axes, ['SSIM','PSNR']):
        ax.scatter(sub['BPP'], sub[ycol], color=col, alpha=0.25, s=10)
        avg = sub.groupby('BPP')[ycol].mean().reset_index().sort_values('BPP')
        ax.plot(avg['BPP'], avg[ycol], color=col, lw=2.5, label=lbl)

axes[0].set_xlabel('BPP'); axes[0].set_ylabel('SSIM')
axes[0].set_title('Rate-Distortion: BPP vs SSIM'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('BPP'); axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('Rate-Distortion: BPP vs PSNR'); axes[1].legend(fontsize=8)

fig.suptitle('Experiment 1 - Rate-Distortion Scatter', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

In [38]:
# CELL 8 - Exp 1: Gain over OMP baseline
omp_k = df1[df1['Method']=='OMP'].groupby('k')[['SSIM','PSNR']].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG)
for ci, mn in enumerate(methods1):
    if mn == 'OMP': continue
    col = ACCENT[ci%10]
    lbl = mn.replace('Spectral','Spec').replace('(','').replace(')','').replace('SIBS1','SIBS')
    sub = df1[df1['Method']==mn].groupby('k')[['SSIM','PSNR']].mean()
    axes[0].plot(sub.index, (sub['SSIM'] - omp_k['SSIM']) / omp_k['SSIM'] * 100,
                 'o-', color=col, lw=2, ms=5, label=lbl)
    axes[1].plot(sub.index, sub['PSNR'] - omp_k['PSNR'],
                 's-', color=col, lw=2, ms=5, label=lbl)

for ax, yl in zip(axes, ['SSIM % gain over OMP', 'PSNR dB gain over OMP']):
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_xlabel('Sparsity k'); ax.set_ylabel(yl); ax.legend(fontsize=8)
axes[0].set_title('SSIM % Gain vs OMP')
axes[1].set_title('PSNR dB Gain vs OMP')

fig.suptitle('Experiment 1 - Improvement over OMP Baseline', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

In [39]:
# CELL 9 - Exp 1: Radar chart + metric correlation heatmap
fig = plt.figure(figsize=(18, 8), facecolor=BG)
ax_radar = fig.add_subplot(1, 2, 1, polar=True)
ax_corr  = fig.add_subplot(1, 2, 2)

# Radar
metrics_r = ['SSIM','PSNR','CR','QR','ETA']
nd  = df1.groupby('Method')[metrics_r].mean()
ndn = (nd - nd.min()) / (nd.max() - nd.min() + 1e-10)
N   = len(metrics_r)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist(); angles += angles[:1]
ax_radar.set_facecolor('#0f1a2e')
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(metrics_r, color=TEXT, fontsize=11)
ax_radar.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_radar.set_yticklabels(['0.25','0.50','0.75','1.00'], color=SUB, fontsize=7)
ax_radar.spines['polar'].set_color(GRID); ax_radar.grid(color=GRID, alpha=0.5)
for ci, mn in enumerate(ndn.index):
    vals = ndn.loc[mn].tolist(); vals += vals[:1]; col = ACCENT[ci%10]
    lbl  = mn.replace('Spectral','Spec').replace('(','').replace(')','').replace('SIBS1','SIBS')
    ax_radar.plot(angles, vals, color=col, lw=2, label=lbl)
    ax_radar.fill(angles, vals, color=col, alpha=0.08)
ax_radar.set_title('Normalised Metric Radar', color=TEXT, fontsize=11, pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.45, 1.2), fontsize=8)

# Correlation heatmap
metric_cols = [c for c in ['SSIM','PSNR','NMSE','CR','QR','ETA','BPP','NNZC'] if c in df1.columns]
corr = df1[metric_cols].corr().round(2)
ax_corr.set_facecolor(PANEL)
cmap2 = LinearSegmentedColormap.from_list('rdbu', ['#cc2244','#0b0f1a','#2266cc'])
sns.heatmap(corr, ax=ax_corr, cmap=cmap2, center=0, annot=True, fmt='.2f',
            linewidths=0.4, linecolor=BG, annot_kws={'size':9,'color':'white'},
            cbar_kws={'shrink':0.8})
ax_corr.set_title('Metric Correlation Matrix', color=TEXT, fontsize=11)

fig.suptitle('Experiment 1 - Radar Chart & Metric Correlations', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

In [40]:
# CELL 10 - Exp 2: Learned atoms metrics + per-image heatmap
methods2 = df2['Method'].unique()

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor=BG)
axes = axes.flatten()
for ax, (col, lab) in zip(axes, [('SSIM','SSIM'), ('PSNR','PSNR (dB)'),
                                   ('NMSE','NMSE'),  ('CR','CR')]):
    for ci, mn in enumerate(methods2):
        sub = df2[df2['Method']==mn].groupby('k')[col].mean()
        ax.plot(sub.index, sub.values, 'o-', color=ACCENT[ci%10], label=mn, lw=2, ms=5)
    ax.set_xlabel('k'); ax.set_ylabel(lab)
    ax.set_title(f'Exp2 - {lab} vs k'); ax.legend(fontsize=7)
fig.suptitle('Experiment 2 - Image-Learned Atom Augmentation', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

piv2 = df2.pivot_table(values='SSIM', index='Method', columns='Image', aggfunc='mean').round(3)
fig, ax = plt.subplots(figsize=(max(14, len(piv2.columns)*0.85), 4), facecolor=BG)
ax.set_facecolor(PANEL)
sns.heatmap(piv2, ax=ax, cmap='YlOrRd', annot=True, fmt='.3f',
            linewidths=0.4, linecolor=BG, annot_kws={'size':6,'color':'black'},
            cbar_kws={'shrink':0.8})
ax.set_title('Exp 2 - Per-Image Mean SSIM', color=TEXT, fontsize=11)
plt.xticks(rotation=40, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [41]:
# CELL 11 - Exp 3: Pre-learned dictionaries grouped bars + pivot heatmaps
dict_names = ['KSVD','MOD','ODL','BKSVD','BSSDL','RBDL']

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
x = np.arange(len(dict_names)); W = 0.25
for ax, metric in zip(axes, ['SSIM','PSNR']):
    ax.set_facecolor(PANEL)
    for bi, (base, col) in enumerate(zip(['OMP','SIBS1','Spectral'], ACCENT[:3])):
        vals = [df3[(df3['Base']==base)&(df3['Dict']==dn)][metric].mean()
                if len(df3[(df3['Base']==base)&(df3['Dict']==dn)])>0 else 0
                for dn in dict_names]
        bars = ax.bar(x + bi*W, vals, W, label=base, color=col, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=6, color=TEXT)
    ax.set_xticks(x+W); ax.set_xticklabels(dict_names, color=SUB)
    ax.set_ylabel(metric); ax.set_title(f'Exp3 - {metric} by Dict x Method')
    ax.legend(fontsize=8)
fig.suptitle('Experiment 3 - Pre-Learned Dictionaries', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
for ax, metric in zip(axes, ['SSIM','PSNR']):
    ax.set_facecolor(PANEL)
    grp = df3.groupby(['Dict','Base'])[metric].mean().unstack('Base').round(4)
    sns.heatmap(grp, ax=ax, cmap='YlOrRd', annot=True, fmt='.4f',
                linewidths=0.5, linecolor=BG, annot_kws={'size':10,'color':'black'},
                cbar_kws={'shrink':0.8})
    ax.set_title(f'Exp3 - {metric}: Dict x Base-Method', color=TEXT, fontsize=11)
    plt.setp(ax.get_xticklabels(), rotation=0)
    plt.setp(ax.get_yticklabels(), rotation=0)
fig.suptitle('Experiment 3 - Dictionary x Encoding Method Pivot', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

In [42]:
# CELL 12 - Exp 4: Rate-distortion with overhead
methods4 = df4['Method'].unique()

fig, axes = plt.subplots(1, 3, figsize=(19, 6), facecolor=BG)
for ci, mn in enumerate(methods4):
    col = ACCENT[ci%10]
    sub  = df4[df4['Method']==mn].groupby('BPP')['SSIM'].mean().reset_index().sort_values('BPP')
    axes[0].plot(sub['BPP'], sub['SSIM'], 'o-', color=col, label=mn, lw=2, ms=5)
    sub2 = df4[df4['Method']==mn].groupby('BPP_TOT')['SSIM'].mean().reset_index().sort_values('BPP_TOT')
    axes[1].plot(sub2['BPP_TOT'], sub2['SSIM'], 'o-', color=col, label=mn, lw=2, ms=5)

axes[0].set_xlabel('Base BPP'); axes[0].set_ylabel('SSIM')
axes[0].set_title('SSIM vs Base BPP'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('Total BPP (+ overhead)'); axes[1].set_ylabel('SSIM')
axes[1].set_title('SSIM vs Total BPP'); axes[1].legend(fontsize=8)

oh = df4.groupby('Method')['BPP_OH'].mean().sort_values()
axes[2].set_facecolor(PANEL)
axes[2].barh(range(len(oh)), oh.values, color=ACCENT[:len(oh)], alpha=0.85)
axes[2].set_yticks(range(len(oh))); axes[2].set_yticklabels(oh.index, fontsize=9)
for i, v in enumerate(oh.values):
    axes[2].text(v+0.005, i, f'{v:.3f}', va='center', color=TEXT, fontsize=9)
axes[2].set_xlabel('Mean BPP Overhead'); axes[2].set_title('Side-Info Overhead')

fig.suptitle('Experiment 4 - Rate-Distortion with Overhead Penalty', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

In [43]:
# CELL 13 - Exp 5: Parameter sensitivity 3x3 grid
fig = plt.figure(figsize=(18, 13), facecolor=BG)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.35)

sens_specs = [
    (df5p, 'Number of PCs (p)', 'SSIM',    'SSIM',       'Vary p -> SSIM',          ACCENT[0], 0, 0),
    (df5p, 'Number of PCs (p)', 'PSNR',    'PSNR (dB)',  'Vary p -> PSNR',          ACCENT[1], 0, 1),
    (df5p, 'Number of PCs (p)', 'EXP_VAR', 'Expl. Var.', 'Vary p -> Explained Var', ACCENT[2], 0, 2),
    (df5d, 'Sampling stride d', 'SSIM',    'SSIM',       'Vary d -> SSIM',          ACCENT[3], 1, 0),
    (df5d, 'Sampling stride d', 'PSNR',    'PSNR (dB)',  'Vary d -> PSNR',          ACCENT[4], 1, 1),
    (df5d, 'Sampling stride d', 'N_PCS',   '# PCs used', 'Vary d -> # PCs',         ACCENT[5], 1, 2),
    (df5r, 'p_ratio',           'SSIM',    'SSIM',       'Vary p_ratio -> SSIM',    ACCENT[6], 2, 0),
    (df5r, 'p_ratio',           'PSNR',    'PSNR (dB)',  'Vary p_ratio -> PSNR',    ACCENT[7], 2, 1),
    (df5r, 'p_ratio',           'N_PCS',   '# PCs used', 'Vary p_ratio -> # PCs',   ACCENT[8], 2, 2),
]
for df_, xlbl, ycol, ylbl, title, color, row, col in sens_specs:
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.tick_params(colors=SUB); ax.grid(color=GRID, alpha=0.5)
    if ycol not in df_.columns:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes); continue
    x = df_.index.astype(float); y = df_[ycol].astype(float)
    ax.plot(x, y, 'o-', color=color, lw=2.5, ms=7, zorder=5)
    ax.fill_between(x, y*0.97, y*1.03, color=color, alpha=0.1)
    bi = y.argmax()
    ax.scatter([x[bi]], [y.iloc[bi]], color='white', s=90, zorder=6)
    ax.annotate(f'{y.iloc[bi]:.3f}', (x[bi], y.iloc[bi]),
                xytext=(5,5), textcoords='offset points', color='white', fontsize=8)
    ax.set_xlabel(xlbl, color=TEXT, fontsize=9)
    ax.set_ylabel(ylbl, color=TEXT, fontsize=9)
    ax.set_title(title, color=TEXT, fontsize=10)

fig.suptitle('Experiment 5 - Parameter Sensitivity  (dot = best value)',
             color=TEXT, fontsize=14, y=1.01)
plt.show()

In [44]:
# CELL 14 - Master comparison across all experiments
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)

for ax, metric in zip(axes, ['SSIM', 'PSNR']):
    ax.set_facecolor(PANEL)
    piv = master.pivot_table(values=metric, index='Method',
                             columns='Experiment', aggfunc='mean')
    piv = piv.sort_values(piv.columns[0], ascending=False)
    x = np.arange(len(piv)); nc = len(piv.columns); w = 0.8 / nc
    for ci, exp in enumerate(piv.columns):
        vals = piv[exp].fillna(0).values
        ax.bar(x + ci*w, vals, w, label=exp, color=ACCENT[ci%10], alpha=0.85)
    ax.set_xticks(x + w*(nc-1)/2)
    ax.set_xticklabels(
        [m.replace('Spectral','Spec').replace('(','').replace(')','').replace('SIBS1','SIBS')
         for m in piv.index],
        rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(metric)
    ax.set_title(f'Master - Mean {metric} by Method & Experiment')
    ax.legend(fontsize=8)

fig.suptitle('Master Comparison - All Experiments', color=TEXT, fontsize=13)
plt.tight_layout(); plt.show()

print('Master table:')
print(master.to_string(index=False))

Master table:
          Method    Experiment     SSIM      PSNR     NMSE       CR       QR       ETA      BPP  NNZC     TIME
             OMP      Exp1-DCT 0.638241 22.933084 0.063318 0.156147 0.098090 15.614695 2.109375     9 0.061507
           SIBS1      Exp1-DCT 0.686383 24.345016 0.047187 0.158808 0.104494 15.880824 2.109375     9 0.058022
Spectral(pr=0.3)      Exp1-DCT 0.813315 33.163592 0.022312 0.164760 0.130391 16.476009 2.250000     9 0.054012
Spectral(pr=0.5)      Exp1-DCT 0.849804 34.816403 0.016363 0.165746 0.135669 16.574621 2.250000     9 0.055720
Spectral(pr=0.8)      Exp1-DCT 0.863220 35.439231 0.015097 0.165904 0.137247 16.590357 2.250000     9 0.058174
        KSVD_aug Exp2-AugLearn 0.767084 29.138999 0.023753 0.163194 0.113486 16.319437 2.250000     9 0.055813
         MOD_aug Exp2-AugLearn 0.739860 28.349381 0.040061 0.155959 0.106690 15.595946 2.250000     9 0.051034
         ODL_aug Exp2-AugLearn 0.766778 27.144795 0.034892 0.158273 0.112653 15.827307 2.250000   

In [45]:
# CELL 15 - Inline display of all saved PNG figures
saved_pngs = sorted([f for f in os.listdir(DATA_DIR)
                     if f.endswith('.png') and not f.startswith('nb_')])
print(f'{len(saved_pngs)} saved PNG figures found:')
for f in saved_pngs:
    print(f'  {f}')

# Thumbnail gallery
ncols = 3
nrows = (len(saved_pngs) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows*5), facecolor=BG)
axes = axes.flatten()
for i, fname in enumerate(saved_pngs):
    img = mpimg.imread(os.path.join(DATA_DIR, fname))
    axes[i].imshow(img); axes[i].axis('off')
    axes[i].set_title(fname.replace('.png',''), color=SUB, fontsize=6)
for j in range(len(saved_pngs), len(axes)):
    axes[j].axis('off')
fig.suptitle('All Saved Figures - Thumbnail Gallery', color=TEXT, fontsize=14)
plt.tight_layout(); plt.show()

# Full-resolution key figures
key_figures = [
    'fig00_dataset_gallery.png',
    'fig01_exp1_metrics_vs_k.png',
    'fig02_exp1_per_image_ssim.png',
    'fig03_exp1_sigt.png',
    'fig07_exp3_dict_bars.png',
    'fig09_exp4_rd_curves.png',
    'fig10_exp5_sensitivity.png',
    'fig11_visual_recons.png',
]
for fname in key_figures:
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path): continue
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(16, 8), facecolor=BG)
    ax.imshow(img); ax.axis('off')
    ax.set_title(fname.replace('.png',''), color=TEXT, fontsize=11, pad=8)
    plt.tight_layout(pad=0); plt.show()

12 saved PNG figures found:
  fig00_dataset_gallery.png
  fig01_exp1_metrics_vs_k.png
  fig02_exp1_per_image_ssim.png
  fig03_exp1_sigt.png
  fig04_exp1_ssim_boxplot.png
  fig05_exp2_metrics_vs_k.png
  fig06_exp2_per_image_ssim.png
  fig07_exp3_dict_bars.png
  fig08_exp3_spectral_per_image.png
  fig09_exp4_rd_curves.png
  fig10_exp5_sensitivity.png
  fig11_visual_recons.png


In [46]:
# CELL 16 - Key findings summary
SEP = '=' * 68
print(SEP)
print('  SPECTRALSIBS - KEY FINDINGS')
print(SEP)

for exp_n, df_, lbl in [(1,df1,'DCT'),(2,df2,'Learned'),(3,df3,'Pre-learned'),(4,df4,'RD')]:
    g = df_.groupby('Method')['SSIM'].mean()
    best_m, best_v, worst_v = g.idxmax(), g.max(), g.min()
    gain = (best_v - worst_v) / worst_v * 100
    print(f'  Exp{exp_n} ({lbl:<12}) best={best_m:<28} SSIM={best_v:.4f}  (+{gain:.1f}% vs worst)')

print()
print('  Experiment 5 - Optimal hyper-parameters:')
for name, df_, col in [('p', df5p,'p'), ('d', df5d,'delta'), ('p_ratio', df5r,'p_ratio')]:
    bi = df_['SSIM'].idxmax()
    print(f'    Best {col:<10} = {bi:<6}  SSIM={df_.loc[bi,"SSIM"]:.4f}  PSNR={df_.loc[bi,"PSNR"]:.2f} dB')

print()
spec_ssim = df3[df3['Base']=='Spectral']['SSIM'].mean()
omp_ssim3 = df3[df3['Base']=='OMP']['SSIM'].mean()
print(f'  Spectral vs OMP (Exp3, all dicts & images):')
print(f'    Spectral={spec_ssim:.4f}  OMP={omp_ssim3:.4f}  Gain={( spec_ssim-omp_ssim3)/omp_ssim3*100:.1f}%')
print(SEP)

  SPECTRALSIBS - KEY FINDINGS
  Exp1 (DCT         ) best=Spectral(pr=0.8)             SSIM=0.8632  (+35.2% vs worst)
  Exp2 (Learned     ) best=KSVD_aug                     SSIM=0.7671  (+281.1% vs worst)
  Exp3 (Pre-learned ) best=Spectral_MOD                 SSIM=0.8933  (+16.0% vs worst)
  Exp4 (RD          ) best=Spectral_p16                 SSIM=0.8682  (+36.0% vs worst)

  Experiment 5 - Optimal hyper-parameters:
    Best p          = 16      SSIM=0.8401  PSNR=28.97 dB
    Best delta      = 1       SSIM=0.8887  PSNR=32.39 dB
    Best p_ratio    = 1.0     SSIM=0.8401  PSNR=28.97 dB

  Spectral vs OMP (Exp3, all dicts & images):
    Spectral=0.8910  OMP=0.7869  Gain=13.2%
